# Clean Pipeline 2 — Harvard-GF Training, Fixed-Threshold Evaluation, and GDP External Evaluation

This notebook contains the final supervised glaucoma model, the locked validation operating point, Harvard-GF test evaluation, uncertainty analysis, explainability, and external Harvard-GDP evaluation.

### Final decision threshold

The operating point used throughout this clean notebook is the validation-selected threshold **0.404529**, corresponding to approximately **80% validation sensitivity**. The previous Youden-J threshold analysis is intentionally excluded.

**Important:** the test set is never used for model selection or threshold selection.

## Part A — Harvard-GF training and evaluation

In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, hashlib, io, zipfile, shutil, random, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, matthews_corrcoef, brier_score_loss,
    log_loss, roc_curve
)

# Mount Google Drive and define the fixed project paths.
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/glaucoma_oct_research')
STAGE10_DIR = PROJECT_DIR / 'results/stage10_protocol'
MANIFEST_PATH = STAGE10_DIR / 'main_experiment_manifest.csv'
SSL_MANIFEST_PATH = STAGE10_DIR / 'ssl_training_manifest.csv'
LEAKAGE_AUDIT_PATH = STAGE10_DIR / 'leakage_audit.csv'
DINO_PATH = PROJECT_DIR / 'checkpoints/stage11_ssl_dino_v2_1/dino_teacher_resnet18_encoder_axis0_ssl_v2.pt'

RUN_TAG = 'stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch'
OUT_DIR = PROJECT_DIR / 'results' / RUN_TAG
CKPT_DIR = PROJECT_DIR / 'checkpoints' / RUN_TAG
DRIVE_CACHE_ROOT = PROJECT_DIR / 'data/processed/stage13_gf_fixed12_cache_axis0'
LOCAL_CACHE_ROOT = Path('/content/stage13_gf_fixed12_cache_axis0')
for folder in (OUT_DIR, CKPT_DIR, DRIVE_CACHE_ROOT, LOCAL_CACHE_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

BEST_PATH = CKPT_DIR / 'best_model.pt'
LATEST_PATH = CKPT_DIR / 'latest_training_state.pt'
HISTORY_PATH = OUT_DIR / 'training_history.csv'
CONFIG_PATH = OUT_DIR / 'training_config.json'
LOCK_PATH = OUT_DIR / 'evaluation_protocol_locked.json'
TEST_DONE_PATH = OUT_DIR / 'test_evaluation_done.json'

required_paths = [MANIFEST_PATH, SSL_MANIFEST_PATH, LEAKAGE_AUDIT_PATH, DINO_PATH]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing required project files:\n' + '\n'.join(missing_paths))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', PROJECT_DIR)
print('Existing DINO checkpoint:', DINO_PATH)
print('Output:', OUT_DIR)
print('Drive cache:', DRIVE_CACHE_ROOT)

In [ ]:
# Read the frozen manifests; do not generate a new split.
manifest = pd.read_csv(MANIFEST_PATH, dtype={'sample_id': str})
ssl_manifest = pd.read_csv(SSL_MANIFEST_PATH, dtype={'sample_id': str})
leakage_audit = pd.read_csv(LEAKAGE_AUDIT_PATH)

required_manifest_columns = {
    'dataset', 'sample_id', 'subject_key', 'protocol_split',
    'glaucoma_label', 'oct_locator', 'rnflt_locator'
}
missing_columns = sorted(required_manifest_columns - set(manifest.columns))
if missing_columns:
    raise ValueError(f'Main manifest is missing columns: {missing_columns}')
if len(manifest) != 4300 or manifest.duplicated(['dataset', 'sample_id']).any():
    raise RuntimeError('The frozen Stage 10 manifest is not the expected 4300 unique samples.')
if leakage_audit.empty or leakage_audit['status'].astype(str).str.upper().ne('PASS').any():
    raise RuntimeError('The saved Stage 10 leakage audit is not fully PASS.')

manifest['dataset'] = manifest['dataset'].astype(str).str.strip().str.lower()
manifest['sample_id'] = manifest['sample_id'].astype(str).str.strip()
manifest['subject_key'] = manifest['subject_key'].astype(str).str.strip()
manifest['protocol_split'] = manifest['protocol_split'].astype(str).str.strip().str.lower()
manifest['glaucoma_label'] = pd.to_numeric(manifest['glaucoma_label'], errors='coerce')

gf = manifest.loc[manifest['dataset'].eq('harvard_gf')].copy()
if len(gf) != 3300 or gf['subject_key'].duplicated().any():
    raise RuntimeError('Expected 3300 distinct Harvard-GF subject keys.')
expected_counts = {'train': 2100, 'validation': 300, 'test': 900}
observed_counts = {str(k): int(v) for k, v in gf['protocol_split'].value_counts().items()}
if observed_counts != expected_counts:
    raise RuntimeError(f'Harvard-GF split mismatch: {observed_counts}; expected {expected_counts}.')
train_val_labels = gf.loc[gf['protocol_split'].isin(['train', 'validation']), 'glaucoma_label']
if train_val_labels.isna().any() or not set(train_val_labels.astype(int).unique()).issubset({0, 1}):
    raise RuntimeError('GF train/validation glaucoma labels must be complete binary targets.')

train_frame = gf.loc[gf['protocol_split'].eq('train')].copy().sort_values('sample_id', kind='stable').reset_index(drop=True)
val_frame = gf.loc[gf['protocol_split'].eq('validation')].copy().sort_values('sample_id', kind='stable').reset_index(drop=True)
test_frame = gf.loc[gf['protocol_split'].eq('test')].copy().sort_values('sample_id', kind='stable').reset_index(drop=True)
train_ids = set(train_frame['subject_key'])
val_ids = set(val_frame['subject_key'])
test_ids = set(test_frame['subject_key'])
if train_ids & val_ids or train_ids & test_ids or val_ids & test_ids:
    raise RuntimeError('Subject overlap exists across train, validation, and test.')

# Confirm the SSL manifest contains only GF-train and GDP ssl_pool samples.
if len(ssl_manifest) != 2600 or 'ssl_role' not in ssl_manifest.columns or 'source_split' not in ssl_manifest.columns:
    raise RuntimeError('Unexpected SSL manifest structure or sample count.')
for forbidden_label in ('glaucoma_label', 'progression_td_pointwise'):
    if forbidden_label in ssl_manifest.columns:
        raise RuntimeError(f'Clinical target {forbidden_label!r} appears in the SSL manifest.')
ssl_manifest['dataset'] = ssl_manifest['dataset'].astype(str).str.strip().str.lower()
ssl_manifest['source_split'] = ssl_manifest['source_split'].astype(str).str.strip().str.lower()
gf_ssl = ssl_manifest.loc[ssl_manifest['dataset'].eq('harvard_gf')]
gdp_ssl = ssl_manifest.loc[ssl_manifest['dataset'].eq('harvard_gdp')]
gf_ssl_ids = set(gf_ssl['sample_id'].astype(str).str.strip())
train_sample_ids = set(train_frame['sample_id'].astype(str))
if len(gf_ssl) != 2100 or gf_ssl_ids != train_sample_ids:
    raise RuntimeError('SSL GF samples do not exactly match the official GF training split.')
if not gf_ssl['source_split'].eq('train').all() or not gf_ssl['ssl_role'].eq('gf_train').all():
    raise RuntimeError('The GF component of SSL was not restricted to GF-train.')
if len(gdp_ssl) != 500 or not gdp_ssl['source_split'].eq('ssl_pool').all():
    raise RuntimeError('The DINO checkpoint provenance does not match the expected GDP ssl_pool.')
if set(val_frame['sample_id']) & gf_ssl_ids or set(test_frame['sample_id']) & gf_ssl_ids:
    raise RuntimeError('GF validation/test identifiers overlap the saved SSL manifest.')
if not {'dataset', 'sample_id', 'oct_locator', 'source_split', 'ssl_role'}.issubset(ssl_manifest.columns):
    raise RuntimeError('SSL manifest is missing required provenance columns.')

print('Protocol audit: PASS')
print('Harvard-GF only for supervised training/validation/test')
print('Split sizes:', {name: len(frame) for name, frame in [('train', train_frame), ('validation', val_frame), ('test', test_frame)]})
for name, frame in [('train', train_frame), ('validation', val_frame)]:
    counts = frame['glaucoma_label'].astype(int).value_counts().sort_index().to_dict()
    print(f'{name} labels (0=non-glaucoma, 1=glaucoma):', counts)
print('SSL exposure: 2100 GF-train + 500 GDP ssl_pool, with no GF validation/test overlap.')
print('Test target values are intentionally not inspected until the final evaluation cell.')

# Define immutable identifiers/signatures used by the cache and the training configuration.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def frame_signature(frame):
    columns = ['dataset', 'sample_id', 'subject_key', 'protocol_split', 'oct_locator', 'rnflt_locator']
    text = frame[columns].astype(str).to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def target_signature(frame):
    text = frame[['sample_id', 'glaucoma_label']].astype(str).to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

TRAIN_SIGNATURE = frame_signature(train_frame)
VAL_SIGNATURE = frame_signature(val_frame)
TEST_SIGNATURE = frame_signature(test_frame)
TRAIN_TARGET_SIGNATURE = target_signature(train_frame)
VAL_TARGET_SIGNATURE = target_signature(val_frame)
DINO_SHA256 = sha256_file(DINO_PATH)

# Persist the protocol audit without inspecting test-label distribution.
audit_record = {
    'status': 'PASS', 'dataset': 'harvard_gf',
    'split_counts': expected_counts,
    'train_label_counts': train_frame['glaucoma_label'].astype(int).value_counts().sort_index().to_dict(),
    'validation_label_counts': val_frame['glaucoma_label'].astype(int).value_counts().sort_index().to_dict(),
    'train_signature': TRAIN_SIGNATURE,
    'validation_signature': VAL_SIGNATURE,
    'test_identifier_signature': TEST_SIGNATURE,
    'ssl_composition': {'gf_train': len(gf_ssl), 'gdp_ssl_pool': len(gdp_ssl)},
    'gf_validation_or_test_in_ssl': False,
    'test_label_distribution_inspected': False,
}
(OUT_DIR / 'data_protocol_audit.json').write_text(json.dumps(audit_record, indent=2), encoding='utf-8')
print('DINO checkpoint SHA-256:', DINO_SHA256)
print('GF train manifest signature:', TRAIN_SIGNATURE)

In [ ]:
from torch.utils.checkpoint import checkpoint as activation_checkpoint

# The cache stores fixed axial slices and raw RNFLT maps, not labels as model inputs.
N_SCANS = 12
BSCAN_SIZE = 200
MODEL_SIZE = 224
MODEL_WIDTH = 320
SCAN_CHUNK_SIZE = 12
BATCH_SIZE = 4
NUM_WORKERS = 0
MAX_EPOCHS = 25
MIN_EPOCHS_BEFORE_EARLY_STOP = 8
EARLY_STOP_PATIENCE = 7
MAIN_LR = 2.0e-4
DINO_LAYER4_LR = 1.0e-5
DINO_UNFREEZE_EPOCH = 4
WEIGHT_DECAY = 1.0e-4
GRAD_CLIP_NORM = 1.0
AMP_ENABLED = DEVICE.type == 'cuda'

# Use fixed centers of 12 equal depth intervals for every GF volume and split.
def choose_bscan_indices(depth, n_scans=N_SCANS):
    edges = np.linspace(0, depth, n_scans + 1, dtype=np.int32)
    indices = ((edges[:-1] + edges[1:] - 1) // 2).astype(np.int64)
    if len(indices) != n_scans or len(np.unique(indices)) != n_scans or np.any(indices < 0) or np.any(indices >= depth):
        raise RuntimeError('Invalid fixed B-scan indices.')
    return indices

SCAN_INDICES = choose_bscan_indices(200)
print('Fixed axis-0 scan indices:', SCAN_INDICES.tolist())

def read_locator_bytes(locator):
    locator = str(locator).strip()
    if locator.startswith('zip://'):
        archive, member = locator[len('zip://'):].split('!', 1)
        with zipfile.ZipFile(archive, 'r') as zf:
            return zf.read(member)
    if locator.startswith('nestedzip://'):
        outer_path, nested_name, member = locator[len('nestedzip://'):].split('!', 2)
        with zipfile.ZipFile(outer_path, 'r') as outer:
            nested_bytes = outer.read(nested_name)
        with zipfile.ZipFile(io.BytesIO(nested_bytes), 'r') as nested:
            return nested.read(member)
    if locator.startswith('file://'):
        locator = locator[len('file://'):]
    with open(locator, 'rb') as handle:
        return handle.read()

def read_npz_fields(locator, fields):
    raw = read_locator_bytes(locator)
    try:
        with np.load(io.BytesIO(raw), allow_pickle=False) as data:
            return {key: np.array(data[key], copy=True) for key in fields if key in data.files}
    finally:
        del raw


def impute_invalid_rnflt_map(raw_map):
    # Repair only non-finite or negative pixels, preserving all valid source values.
    values = np.asarray(raw_map, dtype=np.float32).squeeze()
    if values.ndim != 2:
        raise ValueError(f'Expected a 2D RNFLT map; got shape {values.shape}.')
    valid = np.isfinite(values) & (values >= 0)
    invalid_fraction = float(1.0 - valid.mean())
    if not valid.any():
        raise RuntimeError('RNFLT map contains no valid pixels to use for imputation.')
    if valid.all():
        return values.copy(), invalid_fraction
    filled = values.copy()
    nearest_indices = distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled[~valid] = values[tuple(nearest_indices[:, ~valid])]
    if not np.isfinite(filled).all() or np.any(filled < 0):
        raise RuntimeError('RNFLT imputation did not produce a finite non-negative map.')
    return filled, invalid_fraction

def split_cache_paths(root, split):
    root = Path(root)
    return {
        'oct': root / f'oct_scans_{split}.npy',
        'rnflt': root / f'rnflt_raw_{split}.npy',
        'index': root / f'cache_index_{split}.csv',
        'config': root / f'cache_config_{split}.json',
    }

def expected_cache_config(frame, split):
    return {
        'cache_version': 1,
        'split': split,
        'dataset': 'harvard_gf',
        'frame_signature': frame_signature(frame),
        'sample_count': int(len(frame)),
        'n_scans': N_SCANS,
        'axis_policy': 'native axis-0; volume[index, :, :]; no rotation or flip',
        'scan_indices': SCAN_INDICES.tolist(),
        'oct_shape': [int(len(frame)), N_SCANS, BSCAN_SIZE, BSCAN_SIZE],
        'rnflt_shape': [int(len(frame)), BSCAN_SIZE, BSCAN_SIZE],
        'oct_dtype': 'uint8',
        'rnflt_dtype': 'float32',
    }

def cache_is_valid(root, frame, split):
    paths = split_cache_paths(root, split)
    if not all(path.is_file() for path in paths.values()):
        return False
    try:
        config = json.loads(paths['config'].read_text(encoding='utf-8'))
        if config != expected_cache_config(frame, split):
            return False
        oct_array = np.load(paths['oct'], mmap_mode='r')
        rnflt_array = np.load(paths['rnflt'], mmap_mode='r')
        index_frame = pd.read_csv(paths['index'], dtype={'sample_id': str})
        valid = (
            oct_array.shape == (len(frame), N_SCANS, BSCAN_SIZE, BSCAN_SIZE)
            and oct_array.dtype == np.uint8
            and rnflt_array.shape == (len(frame), BSCAN_SIZE, BSCAN_SIZE)
            and rnflt_array.dtype == np.float32
            and index_frame['sample_id'].astype(str).tolist() == frame['sample_id'].astype(str).tolist()
            and np.array_equal(pd.to_numeric(index_frame['glaucoma_label']).to_numpy(dtype=float), frame['glaucoma_label'].to_numpy(dtype=float))
        )
        del oct_array, rnflt_array
        return bool(valid)
    except Exception:
        return False

def copy_cache_files(source_root, destination_root, split):
    source = split_cache_paths(source_root, split)
    destination = split_cache_paths(destination_root, split)
    Path(destination_root).mkdir(parents=True, exist_ok=True)
    for key in ('oct', 'rnflt', 'index', 'config'):
        tmp = destination[key].with_name(destination[key].name + '.copying')
        shutil.copy2(source[key], tmp)
        os.replace(tmp, destination[key])

def build_split_cache(frame, split, allow_test=False):
    if split == 'test' and not allow_test:
        raise RuntimeError('Test caching is prohibited before validation selection and threshold locking.')
    local_ok = cache_is_valid(LOCAL_CACHE_ROOT, frame, split)
    drive_ok = cache_is_valid(DRIVE_CACHE_ROOT, frame, split)
    if local_ok:
        print(f'{split}: valid local cache found; reusing it.')
        return
    if drive_ok:
        print(f'{split}: restoring existing verified cache from Drive.')
        copy_cache_files(DRIVE_CACHE_ROOT, LOCAL_CACHE_ROOT, split)
        if not cache_is_valid(LOCAL_CACHE_ROOT, frame, split):
            raise RuntimeError(f'{split} cache failed validation after restore.')
        return

    print(f'{split}: building fixed-slice cache from raw GF NPZ files.')
    paths = split_cache_paths(LOCAL_CACHE_ROOT, split)
    for path in paths.values():
        if path.exists():
            path.unlink()
    n = len(frame)
    oct_tmp = paths['oct'].with_name(paths['oct'].stem + '.partial.npy')
    rnflt_tmp = paths['rnflt'].with_name(paths['rnflt'].stem + '.partial.npy')
    oct_mm = np.lib.format.open_memmap(oct_tmp, mode='w+', dtype=np.uint8, shape=(n, N_SCANS, BSCAN_SIZE, BSCAN_SIZE))
    rnflt_mm = np.lib.format.open_memmap(rnflt_tmp, mode='w+', dtype=np.float32, shape=(n, BSCAN_SIZE, BSCAN_SIZE))
    try:
        for i, row in enumerate(tqdm(frame.itertuples(index=False), total=n, desc=f'Cache {split}')):
            row_dict = row._asdict()
            fields = read_npz_fields(row_dict['oct_locator'], {'oct_bscans', 'rnflt'})
            volume = fields.get('oct_bscans')
            if volume is None:
                raise KeyError(f'Missing oct_bscans for GF sample {row_dict["sample_id"]}.')
            if volume.shape != (200, 200, 200) or volume.dtype != np.uint8:
                raise ValueError(f'Unexpected GF OCT shape/dtype for {row_dict["sample_id"]}: {volume.shape}, {volume.dtype}.')
            rnflt = fields.get('rnflt')
            del fields
            if rnflt is None:
                rnflt = read_npz_fields(row_dict['rnflt_locator'], {'rnflt'}).get('rnflt')
            if rnflt is None:
                raise KeyError(f'Missing RNFLT map for GF sample {row_dict["sample_id"]}.')
            rnflt = np.asarray(rnflt, dtype=np.float32).squeeze()
            if rnflt.shape != (200, 200):
                raise ValueError(f'Unexpected GF RNFLT shape for {row_dict["sample_id"]}: {rnflt.shape}.')
            oct_mm[i] = volume[SCAN_INDICES, :, :]
            rnflt_mm[i] = rnflt
            del volume, rnflt
        oct_mm.flush()
        rnflt_mm.flush()
    finally:
        del oct_mm, rnflt_mm
    os.replace(oct_tmp, paths['oct'])
    os.replace(rnflt_tmp, paths['rnflt'])
    index = frame[['dataset', 'sample_id', 'subject_key', 'protocol_split', 'glaucoma_label']].copy()
    index.to_csv(paths['index'], index=False)
    paths['config'].write_text(json.dumps(expected_cache_config(frame, split), indent=2), encoding='utf-8')
    if not cache_is_valid(LOCAL_CACHE_ROOT, frame, split):
        raise RuntimeError(f'{split} cache failed local validation.')
    copy_cache_files(LOCAL_CACHE_ROOT, DRIVE_CACHE_ROOT, split)
    if not cache_is_valid(DRIVE_CACHE_ROOT, frame, split):
        raise RuntimeError(f'{split} cache failed Drive validation after saving.')
    print(f'{split}: cache saved and verified on Drive.')

# Compute RNFLT statistics using valid pixels from GF-train only.
def compute_train_rnflt_stats(train_frame):
    paths = split_cache_paths(LOCAL_CACHE_ROOT, 'train')
    maps = np.load(paths['rnflt'], mmap_mode='r')
    count = 0
    total = 0.0
    total_sq = 0.0
    for start in range(0, len(maps), 32):
        batch = np.asarray(maps[start:start + 32], dtype=np.float32)
        valid = np.isfinite(batch) & (batch >= 0)
        values = batch[valid].astype(np.float64, copy=False)
        count += int(values.size)
        total += float(values.sum())
        total_sq += float(np.square(values).sum())
    del maps
    if count == 0:
        raise RuntimeError('No valid RNFLT pixels were found in GF-train.')
    mean = total / count
    variance = max(total_sq / count - mean ** 2, 0.0)
    std = float(np.sqrt(variance))
    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise RuntimeError('Invalid train-only RNFLT statistics.')
    return {
        'dataset': 'harvard_gf', 'source_split': 'train',
        'train_frame_signature': TRAIN_SIGNATURE,
        'mean': float(mean), 'std': std, 'valid_pixel_count': count,
        'valid_rule': 'finite and >= 0; zero is treated as valid',
    }

print('Data reader and cache functions defined.')

In [ ]:
# Only training and validation inputs are cached here; test data remain untouched.
build_split_cache(train_frame, 'train')
build_split_cache(val_frame, 'validation')

STATS_PATH = OUT_DIR / 'rnflt_stats_gf_train_only.json'
if STATS_PATH.is_file():
    old_stats = json.loads(STATS_PATH.read_text(encoding='utf-8'))
else:
    old_stats = None
if old_stats is not None and old_stats.get('train_frame_signature') == TRAIN_SIGNATURE and old_stats.get('source_split') == 'train':
    RNFLT_STATS = old_stats
    print('Reusing verified train-only RNFLT statistics.')
else:
    RNFLT_STATS = compute_train_rnflt_stats(train_frame)
    STATS_PATH.write_text(json.dumps(RNFLT_STATS, indent=2), encoding='utf-8')

print(json.dumps(RNFLT_STATS, indent=2))
for split, frame in [('train', train_frame), ('validation', val_frame)]:
    paths = split_cache_paths(LOCAL_CACHE_ROOT, split)
    oct_cache = np.load(paths['oct'], mmap_mode='r')
    rnflt_cache = np.load(paths['rnflt'], mmap_mode='r')
    print(f'{split}: OCT cache {oct_cache.shape} {oct_cache.dtype}; RNFLT cache {rnflt_cache.shape} {rnflt_cache.dtype}; rows={len(frame)}')
    del oct_cache, rnflt_cache
print('Cache and train-only RNFLT-statistics audit: PASS.')

In [ ]:
class CachedGFDataset(Dataset):
    def __init__(self, frame, split):
        if split == 'test':
            raise RuntimeError('Test data cannot be used by the training/validation dataset class.')
        self.frame = frame.reset_index(drop=True).copy()
        paths = split_cache_paths(LOCAL_CACHE_ROOT, split)
        if not cache_is_valid(LOCAL_CACHE_ROOT, self.frame, split):
            raise RuntimeError(f'{split} cache is missing or mismatched; run Cell 4 again.')
        self.oct = np.load(paths['oct'], mmap_mode='r')
        self.rnflt = np.load(paths['rnflt'], mmap_mode='r')
        self.labels = self.frame['glaucoma_label'].astype(np.float32).to_numpy()
        self.ids = self.frame['sample_id'].astype(str).to_numpy()
        # Impute each RNFLT map only when it is first requested, then reuse it in later epochs.
        self._rnflt_imputed_cache = {}
        self._rnflt_invalid_fraction = {}
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        oct_value = np.array(self.oct[index], copy=True)
        if index not in self._rnflt_imputed_cache:
            raw_map = np.array(self.rnflt[index], dtype=np.float32, copy=True)
            repaired_map, invalid_fraction = impute_invalid_rnflt_map(raw_map)
            self._rnflt_imputed_cache[index] = repaired_map
            self._rnflt_invalid_fraction[index] = invalid_fraction
        rnflt_value = self._rnflt_imputed_cache[index]
        return {
            'oct_uint8': torch.from_numpy(oct_value),
            'rnflt_imputed': torch.from_numpy(np.array(rnflt_value, dtype=np.float32, copy=True)),
            'rnflt_invalid_fraction': torch.tensor(self._rnflt_invalid_fraction[index], dtype=torch.float32),
            'label': torch.tensor(self.labels[index], dtype=torch.float32),
            'sample_id': self.ids[index],
        }


def preprocess_batch(batch, augment=False):
    oct_uint8 = batch['oct_uint8'].to(DEVICE, non_blocking=True)
    rnflt_imputed = batch['rnflt_imputed'].to(DEVICE, non_blocking=True).float()
    b, s, h, w = oct_uint8.shape
    scans = oct_uint8.reshape(b * s, 1, h, w).float().div_(255.0)
    # Match the original DINO SSL preprocessing: preserve aspect ratio and center-pad GF scans.
    resized = F.interpolate(scans, size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False)
    scans = torch.zeros((b * s, 1, MODEL_SIZE, MODEL_WIDTH), dtype=resized.dtype, device=resized.device)
    left = (MODEL_WIDTH - MODEL_SIZE) // 2
    scans[:, :, :, left:left + MODEL_SIZE] = resized
    scans = scans.reshape(b, s, 1, MODEL_SIZE, MODEL_WIDTH)
    if augment:
        # Apply mild intensity-only augmentation consistently across scans within each volume.
        brightness = torch.empty((b, 1, 1, 1, 1), device=DEVICE).uniform_(0.92, 1.08)
        gamma = torch.empty((b, 1, 1, 1, 1), device=DEVICE).uniform_(0.92, 1.08)
        scans = (scans * brightness).clamp(0, 1).pow(gamma)

    # Resize the already-imputed RNFLT map and normalize it as a single input channel.
    rnflt = rnflt_imputed.unsqueeze(1)
    rnflt = F.interpolate(rnflt, size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False)
    rnflt = ((rnflt - float(RNFLT_STATS['mean'])) / float(RNFLT_STATS['std'])).clamp(-5, 5)
    return scans, rnflt


def build_dino_encoder():
    encoder = models.resnet18(weights=None)
    encoder.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    encoder.fc = nn.Identity()
    checkpoint = torch.load(DINO_PATH, map_location='cpu', weights_only=False)
    if checkpoint.get('training_version') != 'dino_v2_1_parametrized_weight_norm_separate_teacher_fp32_loss':
        raise RuntimeError(f'Unexpected DINO training version: {checkpoint.get("training_version")!r}.')
    if checkpoint.get('architecture') != 'torchvision_resnet18_grayscale' or int(checkpoint.get('input_channels', -1)) != 1:
        raise RuntimeError('DINO architecture/channel metadata does not match grayscale ResNet-18.')
    if list(checkpoint.get('input_canvas_hw', [])) != [224, 320]:
        raise RuntimeError('Expected the existing DINO checkpoint trained with a 224x320 canvas; confirm the checkpoint before using the central GF crop.')
    if not str(checkpoint.get('axis_policy', '')).startswith('axis-0'):
        raise RuntimeError('DINO checkpoint does not document native axis-0 B-scan ordering.')
    result = encoder.load_state_dict(checkpoint['state_dict'], strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f'Strict DINO checkpoint load failed: {result}.')
    return encoder


class Residual2DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.skip = nn.Identity() if in_channels == out_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.GroupNorm(8, out_channels),
        )
    def forward(self, x):
        residual = self.skip(x)
        x = F.gelu(self.norm1(self.conv1(x)))
        return F.gelu(self.norm2(self.conv2(x)) + residual)


class RNFLTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2, bias=False),
            nn.GroupNorm(8, 32), nn.GELU(),
            Residual2DBlock(32, 32),
            Residual2DBlock(32, 64, stride=2),
            Residual2DBlock(64, 128, stride=2),
            Residual2DBlock(128, 128, stride=2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.projection = nn.Sequential(nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10))
    def forward(self, x):
        return self.projection(self.features(x))


class CombinedGFGlaucomaModel(nn.Module):
    def __init__(self, dino_encoder):
        super().__init__()
        self.dino_encoder = dino_encoder
        self.scan_attention = nn.Sequential(nn.Linear(513, 128), nn.Tanh(), nn.Dropout(0.10), nn.Linear(128, 1))
        self.oct_projection = nn.Sequential(nn.Linear(513, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10))
        self.rnflt_encoder = RNFLTEncoder()
        self.fusion = nn.Sequential(
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.10),
        )
        self.glaucoma_head = nn.Linear(64, 1)
        for parameter in self.dino_encoder.parameters():
            parameter.requires_grad_(False)
        self.dino_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.dino_encoder.eval()
        return self

    def set_dino_trainable(self, epoch):
        for parameter in self.dino_encoder.parameters():
            parameter.requires_grad_(False)
        if epoch >= DINO_UNFREEZE_EPOCH:
            for parameter in self.dino_encoder.layer4.parameters():
                parameter.requires_grad_(True)
        self.dino_encoder.eval()

    def encode_oct(self, scans):
        if scans.ndim != 5 or tuple(scans.shape[1:]) != (N_SCANS, 1, MODEL_SIZE, MODEL_WIDTH):
            raise ValueError(f'Expected OCT [B,{N_SCANS},1,{MODEL_SIZE},{MODEL_WIDTH}], got {tuple(scans.shape)}.')
        b, s = scans.shape[:2]
        flat = scans.reshape(b * s, 1, MODEL_SIZE, MODEL_WIDTH)
        chunks = []
        trainable = any(p.requires_grad for p in self.dino_encoder.parameters())
        self.dino_encoder.eval()
        for start in range(0, len(flat), SCAN_CHUNK_SIZE):
            part = flat[start:start + SCAN_CHUNK_SIZE]
            if trainable and torch.is_grad_enabled():
                features = activation_checkpoint(self.dino_encoder, part, use_reentrant=False)
            else:
                with torch.no_grad():
                    features = self.dino_encoder(part)
            chunks.append(features)
        features = torch.cat(chunks, dim=0).reshape(b, s, 512)
        positions = torch.linspace(-1.0, 1.0, s, device=features.device, dtype=features.dtype)
        pos_grid = positions.view(1, s, 1).expand(b, -1, -1)
        score_input = torch.cat([features, pos_grid], dim=-1)
        weights = torch.softmax(self.scan_attention(score_input).squeeze(-1).float(), dim=1).to(features.dtype)
        pooled = torch.sum(features * weights.unsqueeze(-1), dim=1)
        pooled_position = torch.sum(positions.view(1, -1) * weights, dim=1, keepdim=True)
        return self.oct_projection(torch.cat([pooled, pooled_position], dim=1))

    def forward(self, scans, rnflt):
        oct_embedding = self.encode_oct(scans)
        rnflt_embedding = self.rnflt_encoder(rnflt)
        fused = self.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
        return self.glaucoma_head(fused).squeeze(-1)


train_dataset = CachedGFDataset(train_frame, 'train')
val_dataset = CachedGFDataset(val_frame, 'validation')

# Save the frozen design/provenance before training starts.
TRAIN_CONFIG = {
    'run_tag': RUN_TAG,
    'dataset': 'harvard_gf',
    'task': 'binary glaucoma detection only',
    'train_n': len(train_frame), 'validation_n': len(val_frame), 'test_n': len(test_frame),
    'train_signature': TRAIN_SIGNATURE, 'validation_signature': VAL_SIGNATURE, 'test_signature': TEST_SIGNATURE,
    'train_target_signature': TRAIN_TARGET_SIGNATURE, 'validation_target_signature': VAL_TARGET_SIGNATURE,
    'dino_checkpoint': str(DINO_PATH), 'dino_sha256': DINO_SHA256,
    'dino_pretraining_exposure': 'GF train split + GDP ssl_pool, unlabeled; no GF validation/test samples',
    'n_scans': N_SCANS, 'scan_indices': SCAN_INDICES.tolist(), 'axis_policy': 'native axis-0',
    'oct_input': 'GF B-scans resized to 224x224 content and center-padded to 224x320, matching original DINO SSL preprocessing',
    'rnflt_input': 'one channel: RNFLT map imputed lazily on first sample access; only non-finite and negative values are replaced by nearest valid pixels; zero remains valid',
    'rnflt_invalid_pixel_rule': 'finite and >= 0 are valid; nearest-valid imputation is performed lazily in Dataset.__getitem__, only when a sample is first fetched, and cached in RAM for later epochs',
    'rnflt_stats': RNFLT_STATS,
    'loss': 'unweighted BCEWithLogits; GF train classes are close to balanced',
    'model_selection': 'highest validation AUROC, with validation BCE as tie-breaker',
    'test_usage': 'no test images or labels in training, checkpoint selection, or threshold selection',
    'batch_size': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'early_stop_patience': EARLY_STOP_PATIENCE,
    'main_lr': MAIN_LR, 'dino_layer4_lr': DINO_LAYER4_LR, 'dino_unfreeze_epoch': DINO_UNFREEZE_EPOCH,
    'weight_decay': WEIGHT_DECAY, 'seed': SEED,
}
CONFIG_HASH = hashlib.sha256(json.dumps(TRAIN_CONFIG, sort_keys=True).encode('utf-8')).hexdigest()
if CONFIG_PATH.is_file():
    existing = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if existing.get('config_hash') != CONFIG_HASH:
        raise RuntimeError('An existing run has a different configuration. Use a new RUN_TAG rather than overwriting it.')
else:
    CONFIG_PATH.write_text(json.dumps({'config_hash': CONFIG_HASH, **TRAIN_CONFIG}, indent=2), encoding='utf-8')


def make_model():
    return CombinedGFGlaucomaModel(build_dino_encoder()).to(DEVICE)


def get_predictions(model, dataset, split_name):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
    model.eval()
    ids, targets, probs, logits = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Predict {split_name}', leave=False):
            scans, rnflt = preprocess_batch(batch, augment=False)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logit = model(scans, rnflt)
            logit = logit.float()
            ids.extend(list(batch['sample_id']))
            targets.extend(batch['label'].float().cpu().numpy().tolist())
            logits.extend(logit.cpu().numpy().tolist())
            probs.extend(torch.sigmoid(logit).cpu().numpy().tolist())
    return pd.DataFrame({'sample_id': ids, 'y_true': np.asarray(targets, dtype=int), 'logit': logits, 'probability': probs})


def binary_train_loss(logits, labels):
    return F.binary_cross_entropy_with_logits(logits.float(), labels.float())

model = make_model()
main_parameters = [p for name, p in model.named_parameters() if not name.startswith('dino_encoder.')]
optimizer = torch.optim.AdamW(
    [
        {'params': main_parameters, 'lr': MAIN_LR, 'base_lr': MAIN_LR, 'group_name': 'main'},
        {'params': list(model.dino_encoder.layer4.parameters()), 'lr': DINO_LAYER4_LR, 'base_lr': DINO_LAYER4_LR, 'group_name': 'dino_layer4'},
    ],
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

history = []
start_epoch = 1
best_auc = -float('inf')
best_val_loss = float('inf')
best_epoch = 0
bad_epochs = 0

# Resume only from a checkpoint with an identical, recorded configuration.
if LATEST_PATH.is_file():
    latest = torch.load(LATEST_PATH, map_location='cpu', weights_only=False)
    if latest.get('config_hash') != CONFIG_HASH:
        raise RuntimeError('Latest checkpoint configuration mismatch. Use a new RUN_TAG.')
    model.load_state_dict(latest['model_state_dict'], strict=True)
    optimizer.load_state_dict(latest['optimizer_state_dict'])
    #scaler.load_state_dict(latest['scaler_state_dict'])
    saved_scaler_state = latest.get("scaler_state_dict", {})
    if scaler.is_enabled() and saved_scaler_state:
        scaler.load_state_dict(saved_scaler_state)
    elif scaler.is_enabled():
        print("No saved AMP state; using a fresh GradScaler.")
    else:
        print("AMP disabled; skipping GradScaler restoration.")
    history = list(latest.get('history', []))
    start_epoch = int(latest.get('next_epoch', 1))
    if latest.get('rng_state') is not None:
        random.setstate(latest['rng_state']['python'])
        np.random.set_state(latest['rng_state']['numpy'])
        torch.set_rng_state(latest['rng_state']['torch_cpu'])
        if torch.cuda.is_available() and latest['rng_state'].get('torch_cuda') is not None:
            torch.cuda.set_rng_state_all(latest['rng_state']['torch_cuda'])
    best_auc = float(latest.get('best_val_auc', -float('inf')))
    best_val_loss = float(latest.get('best_val_loss', float('inf')))
    best_epoch = int(latest.get('best_epoch', 0))
    bad_epochs = int(latest.get('bad_epochs', 0))
    if latest.get('complete', False):
        print(f'Training already completed; best epoch={best_epoch}, validation AUROC={best_auc:.4f}.')
        start_epoch = MAX_EPOCHS + 1

if start_epoch <= MAX_EPOCHS:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.set_dino_trainable(epoch)
        model.train()
        scale_epoch = epoch - 1
        if scale_epoch < 2:
            lr_scale = (scale_epoch + 1) / 2.0
        else:
            progress = min(max((scale_epoch - 2) / max(1, MAX_EPOCHS - 3), 0.0), 1.0)
            lr_scale = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * progress))
        for group in optimizer.param_groups:
            group['lr'] = group['base_lr'] * lr_scale
        generator = torch.Generator().manual_seed(SEED + epoch * 100003)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                                  num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
        total_loss = 0.0
        total_seen = 0
        progress_bar = tqdm(train_loader, desc=f'Train {epoch}/{MAX_EPOCHS}')
        for batch_idx, batch in enumerate(progress_bar):
            scans, rnflt = preprocess_batch(batch, augment=True)
            labels = batch['label'].to(DEVICE, non_blocking=True).float()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(scans, rnflt)
                loss = binary_train_loss(logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite training loss at epoch={epoch}, batch={batch_idx}.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach().cpu()) * len(labels)
            total_seen += len(labels)
            progress_bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.1e}')
        train_loss = total_loss / max(total_seen, 1)

        val_pred = get_predictions(model, val_dataset, 'validation')
        val_y = val_pred['y_true'].to_numpy(dtype=int)
        val_p = val_pred['probability'].to_numpy(dtype=float)
        val_logit = val_pred['logit'].to_numpy(dtype=float)
        val_auc = float(roc_auc_score(val_y, val_p))
        val_ap = float(average_precision_score(val_y, val_p))
        val_loss = float(log_loss(val_y, np.clip(val_p, 1e-7, 1 - 1e-7), labels=[0, 1]))
        val_acc_05 = float(accuracy_score(val_y, (val_p >= 0.5).astype(int)))
        improved = (val_auc > best_auc + 1e-4) or (abs(val_auc - best_auc) <= 1e-4 and val_loss < best_val_loss)
        if improved:
            best_auc = val_auc
            best_val_loss = val_loss
            best_epoch = epoch
            bad_epochs = 0
            torch.save({
                'status': 'BEST_GF_GLAUCOMA_MODEL',
                'model_state_dict': model.state_dict(),
                'best_epoch': best_epoch,
                'best_val_auc': best_auc,
                'best_val_ap': val_ap,
                'best_val_loss': best_val_loss,
                'config_hash': CONFIG_HASH,
                'config': TRAIN_CONFIG,
                'dino_sha256': DINO_SHA256,
            }, BEST_PATH)
            val_pred.to_csv(OUT_DIR / 'best_model_validation_predictions.csv', index=False)
        else:
            bad_epochs += 1

        record = {
            'epoch': epoch,
            'train_bce': train_loss,
            'validation_bce': val_loss,
            'validation_auroc': val_auc,
            'validation_average_precision': val_ap,
            'validation_accuracy_at_0_5': val_acc_05,
            'lr_main': optimizer.param_groups[0]['lr'],
            'lr_dino_layer4': optimizer.param_groups[1]['lr'],
            'dino_layer4_trainable': epoch >= DINO_UNFREEZE_EPOCH,
            'best_epoch_so_far': best_epoch,
            'best_validation_auroc_so_far': best_auc,
        }
        history = [item for item in history if int(item['epoch']) != epoch] + [record]
        history.sort(key=lambda item: int(item['epoch']))
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        next_epoch = epoch + 1
        torch.save({
            'config_hash': CONFIG_HASH,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'next_epoch': next_epoch,
            'best_val_auc': best_auc,
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch,
            'bad_epochs': bad_epochs,
            'history': history,
            'rng_state': {
                'python': random.getstate(),
                'numpy': np.random.get_state(),
                'torch_cpu': torch.get_rng_state(),
                'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            },
            'complete': bool(epoch == MAX_EPOCHS or (epoch >= MIN_EPOCHS_BEFORE_EARLY_STOP and bad_epochs >= EARLY_STOP_PATIENCE)),
        }, LATEST_PATH)
        print(f'Epoch {epoch}: train BCE={train_loss:.4f}; val AUROC={val_auc:.4f}; val AP={val_ap:.4f}; val BCE={val_loss:.4f}; best={best_auc:.4f} (epoch {best_epoch}).')
        if epoch >= MIN_EPOCHS_BEFORE_EARLY_STOP and bad_epochs >= EARLY_STOP_PATIENCE:
            print(f'Early stopping after epoch {epoch}; best epoch={best_epoch}.')
            break

if not BEST_PATH.is_file():
    raise RuntimeError('No best checkpoint exists. Inspect the training output.')
if not HISTORY_PATH.is_file() and history:
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
print('Training stage finished.')
print('Best checkpoint:', BEST_PATH)
print('History:', HISTORY_PATH)
print('The test set has not been loaded or evaluated.')

## Lock the final validation operating point

In [ ]:
# Lock the predefined validation operating point used by the final analysis.
# The threshold was selected on Harvard-GF validation data to target approximately 80% sensitivity.
# No test labels are used to choose this threshold.

LOCKED_THRESHOLD = 0.404529

checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
if checkpoint.get('config_hash') != CONFIG_HASH or checkpoint.get('dino_sha256') != DINO_SHA256:
    raise RuntimeError('Best checkpoint provenance does not match the current run.')

model = make_model()
model.load_state_dict(checkpoint['model_state_dict'], strict=True)

val_predictions = get_predictions(model, val_dataset, 'validation-final')
val_predictions.to_csv(OUT_DIR / 'final_validation_predictions.csv', index=False)

val_y = val_predictions['y_true'].to_numpy(dtype=int)
val_p = val_predictions['probability'].to_numpy(dtype=float)
val_pred_label = (val_p >= LOCKED_THRESHOLD).astype(int)
val_tn, val_fp, val_fn, val_tp = confusion_matrix(
    val_y, val_pred_label, labels=[0, 1]
).ravel()

val_summary = {
    'dataset': 'harvard_gf',
    'validation_n': int(len(val_y)),
    'validation_positive': int((val_y == 1).sum()),
    'validation_negative': int((val_y == 0).sum()),
    'best_epoch': int(checkpoint['best_epoch']),
    'validation_auroc': float(roc_auc_score(val_y, val_p)),
    'validation_average_precision': float(average_precision_score(val_y, val_p)),
    'validation_bce': float(log_loss(val_y, np.clip(val_p, 1e-7, 1 - 1e-7), labels=[0, 1])),
    'threshold': float(LOCKED_THRESHOLD),
    'threshold_selection': 'validation sensitivity-constrained operating point; approximately 80% sensitivity',
    'sensitivity': float(recall_score(val_y, val_pred_label, pos_label=1, zero_division=0)),
    'specificity': float(val_tn / max(val_tn + val_fp, 1)),
    'accuracy': float(accuracy_score(val_y, val_pred_label)),
    'balanced_accuracy': float(balanced_accuracy_score(val_y, val_pred_label)),
    'precision': float(precision_score(val_y, val_pred_label, pos_label=1, zero_division=0)),
    'f1': float(f1_score(val_y, val_pred_label, pos_label=1, zero_division=0)),
    'mcc': float(matthews_corrcoef(val_y, val_pred_label)),
    'tn': int(val_tn), 'fp': int(val_fp), 'fn': int(val_fn), 'tp': int(val_tp),
}

LOCK_PATH.write_text(json.dumps({
    'status': 'LOCKED_BEFORE_TEST',
    'dataset': 'harvard_gf',
    'threshold': float(LOCKED_THRESHOLD),
    'threshold_method': 'validation sensitivity-constrained operating point (~80% sensitivity)',
    'test_accessed': False,
    'test_manifest_signature': TEST_SIGNATURE,
    'best_checkpoint_sha256': sha256_file(BEST_PATH),
    'config_hash': CONFIG_HASH,
}, indent=2), encoding='utf-8')

print(json.dumps(val_summary, indent=2))
print(f'Locked validation threshold: {LOCKED_THRESHOLD:.6f}')


## Harvard-GF final test evaluation

Run the final test evaluation only after the checkpoint and threshold have been locked.

In [ ]:
if TEST_DONE_PATH.is_file():
    raise RuntimeError(f'Test evaluation is already recorded at {TEST_DONE_PATH}. Read the saved outputs; do not rerun or tune from test.')
if not LOCK_PATH.is_file():
    raise FileNotFoundError('Evaluation protocol is not locked. Run Cell 6 first.')
locked_protocol = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
if locked_protocol.get('status') != 'LOCKED_BEFORE_TEST' or locked_protocol.get('test_accessed') is not False:
    raise RuntimeError('Evaluation protocol is not in the required pre-test state.')
if locked_protocol.get('test_manifest_signature') != TEST_SIGNATURE:
    raise RuntimeError('The locked test identifiers do not match the current frozen manifest.')
if locked_protocol.get('best_checkpoint_sha256') != sha256_file(BEST_PATH):
    raise RuntimeError('Best checkpoint changed after the evaluation protocol was locked.')

# Test images are first read here, after checkpoint and threshold selection are frozen.
build_split_cache(test_frame, 'test', allow_test=True)

class CachedGFTestDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True).copy()
        paths = split_cache_paths(LOCAL_CACHE_ROOT, 'test')
        if not cache_is_valid(LOCAL_CACHE_ROOT, self.frame, 'test'):
            raise RuntimeError('Test cache is missing or does not match the frozen test manifest.')
        self.oct = np.load(paths['oct'], mmap_mode='r')
        self.rnflt = np.load(paths['rnflt'], mmap_mode='r')
        self.labels = self.frame['glaucoma_label'].astype(np.float32).to_numpy()
        self.ids = self.frame['sample_id'].astype(str).to_numpy()
        self._rnflt_imputed_cache = {}
        self._rnflt_invalid_fraction = {}
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        if index not in self._rnflt_imputed_cache:
            raw_map = np.array(self.rnflt[index], dtype=np.float32, copy=True)
            repaired_map, invalid_fraction = impute_invalid_rnflt_map(raw_map)
            self._rnflt_imputed_cache[index] = repaired_map
            self._rnflt_invalid_fraction[index] = invalid_fraction
        return {
            'oct_uint8': torch.from_numpy(np.array(self.oct[index], copy=True)),
            'rnflt_imputed': torch.from_numpy(np.array(self._rnflt_imputed_cache[index], dtype=np.float32, copy=True)),
            'rnflt_invalid_fraction': torch.tensor(self._rnflt_invalid_fraction[index], dtype=torch.float32),
            'label': torch.tensor(self.labels[index], dtype=torch.float32),
            'sample_id': self.ids[index],
        }


def evaluate_metrics(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1 - 1e-7)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        'n': int(len(y_true)),
        'positive': int((y_true == 1).sum()),
        'negative': int((y_true == 0).sum()),
        'prevalence': float(np.mean(y_true)),
        'auroc': float(roc_auc_score(y_true, probabilities)),
        'average_precision': float(average_precision_score(y_true, probabilities)),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, predictions)),
        'sensitivity': float(tp / max(tp + fn, 1)),
        'specificity': float(tn / max(tn + fp, 1)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, predictions)),
        'brier_score': float(brier_score_loss(y_true, probabilities)),
        'log_loss': float(log_loss(y_true, probabilities, labels=[0, 1])),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def stratified_bootstrap_ci(y_true, probabilities, threshold, n_bootstrap=2000, seed=20261012):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    negative_idx = np.flatnonzero(y_true == 0)
    positive_idx = np.flatnonzero(y_true == 1)
    if len(negative_idx) == 0 or len(positive_idx) == 0:
        raise RuntimeError('Test set must contain both classes for stratified bootstrap.')
    rng = np.random.default_rng(seed)
    names = ['auroc', 'average_precision', 'accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'mcc', 'brier_score', 'log_loss']
    values = {name: [] for name in names}
    for _ in tqdm(range(n_bootstrap), desc='Stratified bootstrap CI', leave=False):
        sampled = np.concatenate([
            rng.choice(negative_idx, size=len(negative_idx), replace=True),
            rng.choice(positive_idx, size=len(positive_idx), replace=True),
        ])
        metrics = evaluate_metrics(y_true[sampled], probabilities[sampled], threshold)
        for name in names:
            values[name].append(metrics[name])
    ci = {}
    for name, samples in values.items():
        low, high = np.percentile(np.asarray(samples, dtype=float), [2.5, 97.5])
        ci[name] = {'ci95_low': float(low), 'ci95_high': float(high)}
    return ci


test_dataset = CachedGFTestDataset(test_frame)
model = make_model()
checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
threshold = float(locked_protocol['threshold'])
test_predictions = get_predictions(model, test_dataset, 'TEST_FINAL')
test_predictions['prediction'] = (test_predictions['probability'] >= threshold).astype(int)
test_predictions.to_csv(OUT_DIR / 'best_model_test_predictions.csv', index=False)

y_test = test_predictions['y_true'].to_numpy(dtype=int)
p_test = test_predictions['probability'].to_numpy(dtype=float)
point_metrics = evaluate_metrics(y_test, p_test, threshold)
confidence_intervals = stratified_bootstrap_ci(y_test, p_test, threshold, n_bootstrap=2000)
for name, interval in confidence_intervals.items():
    point_metrics[name + '_ci95_low'] = interval['ci95_low']
    point_metrics[name + '_ci95_high'] = interval['ci95_high']

final_report = {
    'status': 'TEST_EVALUATION_COMPLETE',
    'dataset': 'harvard_gf',
    'test_signature': TEST_SIGNATURE,
    'best_checkpoint_sha256': sha256_file(BEST_PATH),
    'best_epoch_selected_by_validation': int(checkpoint['best_epoch']),
    'locked_threshold_from_validation': threshold,
    'threshold_method': locked_protocol['threshold_selection'],
    'test_metrics': point_metrics,
    'confidence_intervals': confidence_intervals,
    'bootstrap': {'method': 'stratified bootstrap within each class', 'replicates': 2000, 'confidence_level': 0.95, 'seed': 20261012},
    'test_was_used_for_model_selection_or_threshold_tuning': False,
    'rnflt_preprocessing': 'single channel; non-finite and negative pixels imputed with nearest valid pixel on first access, cached for reuse; zero retained as valid',
    'clinical_use': 'research only; not a clinically validated diagnostic device',
}
metrics_path = OUT_DIR / 'best_model_test_metrics.json'
metrics_path.write_text(json.dumps(final_report, indent=2), encoding='utf-8')
pd.DataFrame([point_metrics]).to_csv(OUT_DIR / 'best_model_test_metrics.csv', index=False)
TEST_DONE_PATH.write_text(json.dumps({
    'status': 'TEST_EVALUATION_COMPLETE',
    'test_signature': TEST_SIGNATURE,
    'best_checkpoint_sha256': sha256_file(BEST_PATH),
    'test_predictions_path': str(OUT_DIR / 'best_model_test_predictions.csv'),
    'test_metrics_path': str(metrics_path),
    'completed_at_unix': time.time(),
}, indent=2), encoding='utf-8')

print(json.dumps(final_report, indent=2))
print('\nSaved outputs:')
print('Predictions:', OUT_DIR / 'best_model_test_predictions.csv')
print('Metrics JSON:', metrics_path)
print('Metrics CSV:', OUT_DIR / 'best_model_test_metrics.csv')
print('Do not tune or select a model based on these test results.')

## Harvard-GF probability-based performance curves

In [ ]:
# Plot calibration, ROC, and precision-recall curves from saved test predictions.
# No model inference, retraining, or threshold tuning is performed.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss,
    log_loss,
    auc,
)

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/glaucoma_oct_research/results/"
    "stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch"
)

PREDICTIONS_PATH = RUN_DIR / "best_model_test_predictions.csv"
PLOT_DIR = RUN_DIR / "test_curves"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(
        f"Saved test predictions not found: {PREDICTIONS_PATH}"
    )

df = pd.read_csv(PREDICTIONS_PATH)

required_columns = {"y_true", "probability"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Prediction CSV must contain columns: {required_columns}"
    )

y_true = pd.to_numeric(df["y_true"], errors="raise").to_numpy(dtype=int)
y_prob = pd.to_numeric(df["probability"], errors="raise").to_numpy(dtype=float)

if len(y_true) == 0 or len(y_true) != len(y_prob):
    raise ValueError("The labels and probabilities are empty or mismatched.")

if not np.isfinite(y_prob).all() or not np.isin(y_true, [0, 1]).all():
    raise ValueError("Labels or probabilities contain invalid values.")

if ((y_prob < 0) | (y_prob > 1)).any():
    raise ValueError("Predicted probabilities must be between 0 and 1.")

if len(np.unique(y_true)) != 2:
    raise ValueError("Both glaucoma-negative and glaucoma-positive cases are required.")

# Calculate threshold-independent test metrics.
test_auroc = roc_auc_score(y_true, y_prob)
test_ap = average_precision_score(y_true, y_prob)
test_brier = brier_score_loss(y_true, y_prob)
test_log_loss = log_loss(
    y_true, np.clip(y_prob, 1e-7, 1 - 1e-7), labels=[0, 1]
)
prevalence = float(np.mean(y_true))

# 1. Calibration curve with 95% Wilson confidence intervals

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

N_BINS = 5
Z = 1.959963984540054  # 95% two-sided confidence level

# Use the same uniform binning as sklearn's calibration_curve.
bin_edges = np.linspace(0.0, 1.0, N_BINS + 1)
bin_ids = np.searchsorted(bin_edges[1:-1], y_prob)

bin_count = np.bincount(bin_ids, minlength=N_BINS)
sum_predicted = np.bincount(
    bin_ids, weights=y_prob, minlength=N_BINS
)
sum_positive = np.bincount(
    bin_ids, weights=y_true, minlength=N_BINS
)

# Empty bins cannot contribute a calibration point.
valid = bin_count > 0

n = bin_count[valid].astype(float)
mean_predicted = sum_predicted[valid] / n
fraction_positive = sum_positive[valid] / n

# Wilson confidence interval for the observed positive fraction in each bin.
denominator = 1.0 + (Z ** 2) / n

center = (
    fraction_positive + (Z ** 2) / (2.0 * n)
) / denominator

half_width = (
    Z
    * np.sqrt(
        fraction_positive * (1.0 - fraction_positive) / n
        + (Z ** 2) / (4.0 * n ** 2)
    )
    / denominator
)

lower_ci = np.clip(center - half_width, 0.0, 1.0)
upper_ci = np.clip(center + half_width, 0.0, 1.0)

# Sort points by their mean predicted probability.
order = np.argsort(mean_predicted)
mean_predicted = mean_predicted[order]
fraction_positive = fraction_positive[order]
lower_ci = lower_ci[order]
upper_ci = upper_ci[order]
n = n[order]

fig, ax = plt.subplots(figsize=(7.5, 6.5))

# Perfect calibration reference.
ax.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1.5,
    label="Perfect calibration",
)

# Confidence band across the plotted bin estimates.
ax.fill_between(
    mean_predicted,
    lower_ci,
    upper_ci,
    alpha=0.20,
    label="95% Wilson confidence interval",
)

# Calibration curve.
ax.plot(
    mean_predicted,
    fraction_positive,
    marker="o",
    linewidth=2,
    markersize=6,
    label="Model",
)

# Show the interval at each individual bin estimate.
ax.errorbar(
    mean_predicted,
    fraction_positive,
    yerr=np.vstack([
        fraction_positive - lower_ci,
        upper_ci - fraction_positive,
    ]),
    fmt="none",
    capsize=4,
    elinewidth=1.3,
)

ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="Mean predicted probability",
    ylabel="Observed fraction of glaucoma-positive cases",
    title="Calibration Curve with 95% Confidence Intervals",
)

ax.grid(alpha=0.25)
ax.legend(loc="upper left")
fig.tight_layout()

PLOT_DIR = Path(PLOT_DIR)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig.savefig(
    PLOT_DIR / "calibration_curve_with_95CI.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    PLOT_DIR / "calibration_curve_with_95CI.pdf",
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

# Save the bin-level calibration estimates and confidence intervals.
calibration_results = pd.DataFrame({
    "mean_predicted_probability": mean_predicted,
    "observed_positive_fraction": fraction_positive,
    "ci95_lower": lower_ci,
    "ci95_upper": upper_ci,
    "n_in_bin": n.astype(int),
})

calibration_results.to_csv(
    PLOT_DIR / "calibration_curve_bin_statistics.csv",
    index=False,
)

print("Calibration curve with 95% Wilson confidence intervals saved.")
print(f"Output directory: {PLOT_DIR}")

# 2. ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    fpr, tpr,
    linewidth=2,
    label=f"Model (AUROC = {roc_auc:.3f})",
)
ax.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    label="Chance level (AUROC = 0.500)",
)

ax.set(
    xlim=(0, 1),
    ylim=(0, 1.02),
    xlabel="False Positive Rate (1 - Specificity)",
    ylabel="True Positive Rate (Sensitivity)",
    title="Receiver Operating Characteristic (ROC)",
)
ax.grid(alpha=0.25)
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(PLOT_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
fig.savefig(PLOT_DIR / "roc_curve.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# 3. Precision-recall curve
precision, recall, _ = precision_recall_curve(y_true, y_prob)
average_precision = average_precision_score(y_true, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(
    recall, precision,
    linewidth=2,
    label=f"Model (Average Precision = {average_precision:.3f})",
)
ax.axhline(
    prevalence,
    linestyle="--",
    color="gray",
    label=f"Positive prevalence = {prevalence:.3f}",
)

ax.set(
    xlim=(0, 1),
    ylim=(0, 1.02),
    xlabel="Recall (Sensitivity)",
    ylabel="Precision (Positive Predictive Value)",
    title="Precision-Recall Curve",
)
ax.grid(alpha=0.25)
ax.legend(loc="lower left")
fig.tight_layout()

fig.savefig(PLOT_DIR / "precision_recall_curve.png", dpi=300, bbox_inches="tight")
fig.savefig(PLOT_DIR / "precision_recall_curve.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# Save a compact record of the metrics calculated from the saved predictions.
summary = {
    "dataset": "harvard_gf",
    "n_test": int(len(y_true)),
    "positive_cases": int(y_true.sum()),
    "negative_cases": int(len(y_true) - y_true.sum()),
    "prevalence": prevalence,
    "auroc": float(test_auroc),
    "average_precision": float(test_ap),
    "brier_score": float(test_brier),
    "log_loss": float(test_log_loss),
    "calibration_bins": 10,
    "calibration_strategy": "uniform",
    "source_predictions": str(PREDICTIONS_PATH),
}

with open(PLOT_DIR / "curve_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nAll figures and metrics saved to:\n{PLOT_DIR}")

## Harvard-GF explainability

In [ ]:

# Explainability for correctly classified glaucoma-positive test cases.
# Requires the trained model, test_dataset, preprocess_batch, and model definitions.

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

# Use the existing project output directory.
XAI_DIR = OUT_DIR / "explainability"
XAI_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 5
TARGET_LAYER = model.dino_encoder.layer4[-1]

# Load the saved test predictions; do not change the threshold or model.
PREDICTIONS_PATH = (
    OUT_DIR / "best_model_test_predictions.csv"
)
if not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(
        f"Test predictions not found: {PREDICTIONS_PATH}"
    )

predictions = pd.read_csv(PREDICTIONS_PATH)

# Select true-positive cases using the locked validation threshold.
locked = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
threshold = float(locked["threshold"])

tp = predictions[
    (predictions["y_true"] == 1)
    & (predictions["probability"] >= threshold)
].sort_values("probability", ascending=False).head(TOP_K)

if tp.empty:
    raise RuntimeError("No true-positive test cases were found.")

def normalize_sample_id(value):
    s = str(value).strip()
    try:
        number = float(s)
        if np.isfinite(number) and number.is_integer():
            return str(int(number))
    except (ValueError, TypeError):
        pass
    return s


# Normalize IDs from both sources to the same representation.
dataset_ids = [normalize_sample_id(x) for x in test_dataset.ids]
id_to_index = {sample_id: i for i, sample_id in enumerate(dataset_ids)}

tp["sample_id"] = tp["sample_id"].map(normalize_sample_id)
missing_ids = set(tp["sample_id"].astype(str)) - set(id_to_index)
if missing_ids:
    raise RuntimeError(
        "Some selected prediction IDs are absent from test_dataset. "
        "Check that the predictions and test cache belong to the same run."
    )

model.eval()
device = next(model.parameters()).device


def prepare_one_sample(index):
    sample = test_dataset[index]

    batch = {
        key: value.unsqueeze(0) if torch.is_tensor(value) else [value]
        for key, value in sample.items()
    }
    scans, rnflt = preprocess_batch(batch, augment=False)
    return scans, rnflt


def forward_with_oct_gradients(scans, rnflt):
    """Run the OCT encoder with gradients enabled for Grad-CAM."""
    b, s = scans.shape[:2]
    flat = scans.reshape(
    b * s, 1, MODEL_SIZE, MODEL_WIDTH).detach().requires_grad_(True)

    feature_maps = []
    for start in range(0, len(flat), SCAN_CHUNK_SIZE):
        x = flat[start:start + SCAN_CHUNK_SIZE]

        # Explicit ResNet forward so gradients reach layer4.
        x = model.dino_encoder.conv1(x)
        x = model.dino_encoder.bn1(x)
        x = model.dino_encoder.relu(x)
        x = model.dino_encoder.maxpool(x)
        x = model.dino_encoder.layer1(x)
        x = model.dino_encoder.layer2(x)
        x = model.dino_encoder.layer3(x)
        x = model.dino_encoder.layer4(x)
        feature_maps.append(x)

    fmap = torch.cat(feature_maps, dim=0)
    pooled_features = F.adaptive_avg_pool2d(fmap, 1).flatten(1)
    features = pooled_features.reshape(b, s, 512)

    positions = torch.linspace(
        -1.0, 1.0, s, device=features.device, dtype=features.dtype
    )
    pos_grid = positions.view(1, s, 1).expand(b, -1, -1)
    score_input = torch.cat([features, pos_grid], dim=-1)

    attention_logits = model.scan_attention(score_input).squeeze(-1)
    attention = torch.softmax(attention_logits.float(), dim=1).to(features.dtype)

    pooled = torch.sum(features * attention.unsqueeze(-1), dim=1)
    pooled_position = torch.sum(
        positions.view(1, -1) * attention, dim=1, keepdim=True
    )
    oct_embedding = model.oct_projection(
        torch.cat([pooled, pooled_position], dim=1)
    )

    rnflt_embedding = model.rnflt_encoder(rnflt)
    fused = model.fusion(
        torch.cat([oct_embedding, rnflt_embedding], dim=1)
    )
    logit = model.glaucoma_head(fused).squeeze(-1)

    return logit, fmap, attention


def make_gradcam(feature_maps, score):
    gradients = torch.autograd.grad(
        outputs=score.sum(),
        inputs=feature_maps,
        retain_graph=False,
        create_graph=False,
    )[0]

    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * feature_maps).sum(dim=1, keepdim=True))
    cam = F.interpolate(
        cam,
        size=(MODEL_SIZE, MODEL_WIDTH),
        mode="bilinear",
        align_corners=False,
    )
    cam = cam[:, 0]
    cam_min = cam.amin(dim=(1, 2), keepdim=True)
    cam_max = cam.amax(dim=(1, 2), keepdim=True)
    return (cam - cam_min) / (cam_max - cam_min + 1e-8)


def make_rnflt_integrated_gradients(scans, rnflt, steps=32):
    """Attribute the positive glaucoma logit to RNFLT input pixels."""
    baseline = torch.zeros_like(rnflt)
    alphas = torch.linspace(0, 1, steps, device=device)
    accumulated = torch.zeros_like(rnflt)

    for alpha in alphas:
        x = (baseline + alpha * (rnflt - baseline)).detach()
        x.requires_grad_(True)

        with torch.enable_grad():
            score = model(scans, x)
            gradient = torch.autograd.grad(score.sum(), x)[0]

        accumulated += gradient.detach()

    attribution = (rnflt - baseline) * accumulated / steps
    attribution = attribution.abs()[0, 0].cpu().numpy()
    attribution /= attribution.max() + 1e-8
    return attribution


def original_scan_canvas(scan_uint8):
    """Reproduce the model's resize and center-padding for display."""
    scan = torch.as_tensor(scan_uint8, device=device).float().div(255.0)
    scan = scan.reshape(1, 1, *scan.shape[-2:])
    resized = F.interpolate(
        scan, size=(MODEL_SIZE, MODEL_SIZE),
        mode="bilinear", align_corners=False
    )
    canvas = torch.zeros(
        (1, 1, MODEL_SIZE, MODEL_WIDTH),
        device=device, dtype=resized.dtype
    )
    left = (MODEL_WIDTH - MODEL_SIZE) // 2
    canvas[:, :, :, left:left + MODEL_SIZE] = resized
    return canvas[0, 0].detach().cpu().numpy()


records = []

for _, row in tp.iterrows():
    sample_id = str(row["sample_id"])
    index = id_to_index[sample_id]
    scans, rnflt = prepare_one_sample(index)

    # Capture intermediate activations for the selected ResNet layer.
    captured = {}

    def save_activation(module, inputs, output):
        captured["activation"] = output

    handle = TARGET_LAYER.register_forward_hook(save_activation)

    try:
        scans = scans.to(device)
        rnflt = rnflt.to(device)

        # Grad-CAM needs gradients even though the DINO encoder is frozen.
        with torch.enable_grad():
            logit, feature_maps, attention = forward_with_oct_gradients(
                scans, rnflt
            )
            probability = torch.sigmoid(logit).item()
            gradcam = make_gradcam(feature_maps, logit).detach().cpu().numpy()

        # RNFLT attribution uses the full model's output.
        rnflt_ig = make_rnflt_integrated_gradients(scans, rnflt)

    finally:
        handle.remove()

    sample = test_dataset[index]
    raw_scans = sample["oct_uint8"].cpu().numpy()
    scan_weights = attention[0].detach().cpu().numpy()

    # Select the scan with the largest attention weight for the main display.
    best_scan = int(np.argmax(scan_weights))
    scan_image = original_scan_canvas(raw_scans[best_scan])

    # The CAM is computed in the DINO canvas coordinate system.
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

    axes[0].imshow(scan_image, cmap="gray")
    axes[0].set_title(f"OCT B-scan #{best_scan + 1}")
    axes[0].axis("off")

    axes[1].imshow(scan_image, cmap="gray")
    axes[1].imshow(gradcam[best_scan], cmap="jet", alpha=0.45, vmin=0, vmax=1)
    axes[1].set_title("OCT Grad-CAM")
    axes[1].axis("off")

    rnflt_display = rnflt[0, 0].detach().cpu().numpy()
    axes[2].imshow(rnflt_display, cmap="viridis")
    axes[2].set_title("RNFLT input map")
    axes[2].axis("off")

    axes[3].imshow(rnflt_ig, cmap="magma", vmin=0, vmax=1)
    axes[3].set_title("RNFLT Integrated Gradients")
    axes[3].axis("off")

    fig.suptitle(
        f"Sample {sample_id} | True glaucoma | "
        f"P={probability:.3f} | threshold={threshold:.3f}",
        fontsize=12
    )
    fig.tight_layout()

    safe_id = "".join(c if c.isalnum() or c in "-_" else "_" for c in sample_id)
    fig.savefig(XAI_DIR / f"{safe_id}_explanation.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # Show how the model distributes attention over the 12 sampled scans.
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.bar(np.arange(1, len(scan_weights) + 1), scan_weights)
    ax.set_xlabel("Sampled B-scan position")
    ax.set_ylabel("Attention weight")
    ax.set_title(f"Scan attention — {sample_id}")
    ax.set_xticks(np.arange(1, len(scan_weights) + 1))
    fig.tight_layout()
    fig.savefig(XAI_DIR / f"{safe_id}_attention.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    records.append({
        "sample_id": sample_id,
        "true_label": int(row["y_true"]),
        "predicted_probability": probability,
        "locked_threshold": threshold,
        "highest_attention_scan_1based": best_scan + 1,
        "highest_attention_weight": float(scan_weights[best_scan]),
    })

pd.DataFrame(records).to_csv(
    XAI_DIR / "selected_true_positive_cases.csv", index=False
)

print(f"Saved explainability outputs to: {XAI_DIR}")


In [ ]:
# Occlusion Sensitivity for the exact cases used in the previous XAI cell.
# This cell does not select new samples or modify the trained model.

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Reuse the exact sample manifest from the previous explainability run.
XAI_DIR = OUT_DIR / "explainability"
CASES_PATH = XAI_DIR / "selected_true_positive_cases.csv"

if not CASES_PATH.is_file():
    raise FileNotFoundError(
        f"{CASES_PATH} not found. First run the previous Grad-CAM/Integrated "
        "Gradients cell to completion."
    )

selected_cases = pd.read_csv(CASES_PATH)

required_columns = {
    "sample_id",
    "highest_attention_scan_1based",
}
if not required_columns.issubset(selected_cases.columns):
    raise RuntimeError(
        f"The saved case manifest must contain: {required_columns}"
    )

# Normalize IDs consistently with the earlier cell.
def normalize_sample_id(value):
    value = str(value).strip()
    try:
        number = float(value)
        if np.isfinite(number) and number.is_integer():
            return str(int(number))
    except (ValueError, TypeError):
        pass
    return value


dataset_ids = [normalize_sample_id(x) for x in test_dataset.ids]
id_to_index = {sample_id: i for i, sample_id in enumerate(dataset_ids)}

selected_cases["sample_id"] = selected_cases["sample_id"].map(
    normalize_sample_id
)

missing_ids = set(selected_cases["sample_id"]) - set(id_to_index)
if missing_ids:
    raise RuntimeError(
        f"Selected IDs do not match test_dataset: {sorted(missing_ids)}"
    )

# Keep the original model and preprocessing unchanged.
model.eval()
device = next(model.parameters()).device

PATCH_SIZE = 32
OCCLUSION_BATCH_SIZE = 4

if MODEL_SIZE % PATCH_SIZE != 0:
    raise ValueError("PATCH_SIZE must divide MODEL_SIZE for this grid.")

LEFT = (MODEL_WIDTH - MODEL_SIZE) // 2

# A 7 x 7 grid of patches covers each 224 x 224 input region.
patches = [
    (y, x, min(y + PATCH_SIZE, MODEL_SIZE),
     min(x + PATCH_SIZE, MODEL_SIZE))
    for y in range(0, MODEL_SIZE, PATCH_SIZE)
    for x in range(0, MODEL_SIZE, PATCH_SIZE)
]


def load_sample_inputs(dataset_index):
    sample = test_dataset[dataset_index]
    batch = {
        key: value.unsqueeze(0) if torch.is_tensor(value) else [value]
        for key, value in sample.items()
    }
    scans, rnflt = preprocess_batch(batch, augment=False)
    return scans.to(device), rnflt.to(device)


@torch.no_grad()
def get_logit(scans, rnflt):
    return model(scans, rnflt).float().reshape(-1)


@torch.no_grad()
def compute_occlusion_logits(
    modality, scans, rnflt, scan_index, patch_grid, fill_value
):
    """
    Return model logits after occluding each grid patch.
    The other input modality remains unchanged.
    """
    results = []

    for start in range(0, len(patch_grid), OCCLUSION_BATCH_SIZE):
        current_patches = patch_grid[
            start:start + OCCLUSION_BATCH_SIZE
        ]
        batch_n = len(current_patches)

        if modality == "oct":
            scan_batch = scans.expand(
                batch_n, -1, -1, -1, -1
            ).clone()
            rnflt_batch = rnflt.expand(
                batch_n, -1, -1, -1
            ).contiguous()

            for j, (y1, x1, y2, x2) in enumerate(current_patches):
                # Occlude only the B-scan displayed in the previous Grad-CAM figure.
                scan_batch[
                    j, scan_index, 0,
                    y1:y2, LEFT + x1:LEFT + x2
                ] = fill_value

        elif modality == "rnflt":
            scan_batch = scans.expand(
                batch_n, -1, -1, -1, -1
            ).contiguous()
            rnflt_batch = rnflt.expand(
                batch_n, -1, -1, -1
            ).clone()

            for j, (y1, x1, y2, x2) in enumerate(current_patches):
                # Zero corresponds to the training mean after RNFLT normalization.
                rnflt_batch[j, 0, y1:y2, x1:x2] = 0.0

        else:
            raise ValueError(f"Unknown modality: {modality}")

        batch_logits = get_logit(scan_batch, rnflt_batch)
        results.extend(batch_logits.detach().cpu().numpy().tolist())

    return np.asarray(results, dtype=np.float32)


def make_delta_map(base_logit, occluded_logits, patch_grid):
    """
    Positive values mean occluding a patch lowered the glaucoma logit.
    Negative values mean occluding a patch raised the glaucoma logit.
    """
    deltas = base_logit - occluded_logits
    result = np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=np.float32)

    for delta, (y1, x1, y2, x2) in zip(deltas, patch_grid):
        result[y1:y2, x1:x2] = delta

    return result, deltas


records = []

for _, case in selected_cases.iterrows():
    sample_id = case["sample_id"]
    dataset_index = id_to_index[sample_id]

    # Reuse the same scan selected for display in the prior Grad-CAM analysis.
    scan_index = int(case["highest_attention_scan_1based"]) - 1
    if not 0 <= scan_index < N_SCANS:
        raise RuntimeError(
            f"Invalid scan index {scan_index} for sample {sample_id}."
        )

    scans, rnflt = load_sample_inputs(dataset_index)

    # Baseline prediction with both original inputs.
    base_logit = float(get_logit(scans, rnflt).item())
    base_probability = float(torch.sigmoid(torch.tensor(base_logit)).item())

    # OCT baseline: mean intensity of the valid 224 x 224 image content.
    oct_image = scans[
        0, scan_index, 0, :, LEFT:LEFT + MODEL_SIZE
    ].detach().cpu().numpy()
    oct_fill = scans[
        0, scan_index, 0, :, LEFT:LEFT + MODEL_SIZE
    ].mean()

    # Assess how each OCT patch affects the model output.
    oct_occluded_logits = compute_occlusion_logits(
        "oct", scans, rnflt, scan_index, patches, oct_fill
    )
    oct_delta_map, oct_deltas = make_delta_map(
        base_logit, oct_occluded_logits, patches
    )

    # Assess how each RNFLT patch affects the model output.
    rnflt_occluded_logits = compute_occlusion_logits(
        "rnflt", scans, rnflt, scan_index, patches, 0.0
    )
    rnflt_delta_map, rnflt_deltas = make_delta_map(
        base_logit, rnflt_occluded_logits, patches
    )

    rnflt_image = rnflt[0, 0].detach().cpu().numpy()

    # Use the same signed color scale for both maps of this sample.
    combined = np.concatenate([
        oct_delta_map.ravel(),
        rnflt_delta_map.ravel()
    ])
    color_limit = float(np.percentile(np.abs(combined), 98))
    color_limit = max(color_limit, 1e-6)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

    axes[0].imshow(oct_image, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"OCT B-scan #{scan_index + 1}")
    axes[0].axis("off")

    axes[1].imshow(oct_image, cmap="gray", vmin=0, vmax=1)
    axes[1].imshow(
        oct_delta_map,
        cmap="coolwarm",
        alpha=0.60,
        vmin=-color_limit,
        vmax=color_limit,
        interpolation="nearest",
    )
    axes[1].set_title("OCT Occlusion Sensitivity")
    axes[1].axis("off")

    axes[2].imshow(rnflt_image, cmap="viridis")
    axes[2].set_title("RNFLT input")
    axes[2].axis("off")

    axes[3].imshow(rnflt_image, cmap="viridis")
    overlay = axes[3].imshow(
        rnflt_delta_map,
        cmap="coolwarm",
        alpha=0.60,
        vmin=-color_limit,
        vmax=color_limit,
        interpolation="nearest",
    )
    axes[3].set_title("RNFLT Occlusion Sensitivity")
    axes[3].axis("off")

    fig.colorbar(
        overlay,
        ax=[axes[1], axes[3]],
        fraction=0.035,
        pad=0.03,
        label="Baseline logit − occluded logit",
    )

    fig.suptitle(
        f"Sample {sample_id} | "
        f"P(glaucoma)={base_probability:.3f} | "
        f"Occlusion patch={PATCH_SIZE}×{PATCH_SIZE}",
        fontsize=12,
    )
    fig.tight_layout()

    safe_id = "".join(
        c if c.isalnum() or c in "-_" else "_"
        for c in str(sample_id)
    )

    fig.savefig(
        XAI_DIR / f"{safe_id}_occlusion_sensitivity.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    # Save numerical maps for later reproducible analysis.
    np.savez_compressed(
        XAI_DIR / f"{safe_id}_occlusion_maps.npz",
        oct_delta_logit=oct_delta_map,
        rnflt_delta_logit=rnflt_delta_map,
        patch_size=PATCH_SIZE,
        selected_scan_1based=scan_index + 1,
        baseline_logit=base_logit,
        baseline_probability=base_probability,
    )

    records.append({
        "sample_id": sample_id,
        "selected_scan_1based": scan_index + 1,
        "baseline_logit": base_logit,
        "baseline_probability": base_probability,
        "max_oct_logit_drop": float(np.max(oct_delta_map)),
        "max_rnflt_logit_drop": float(np.max(rnflt_delta_map)),
        "mean_abs_oct_effect": float(np.mean(np.abs(oct_deltas))),
        "mean_abs_rnflt_effect": float(np.mean(np.abs(rnflt_deltas))),
        "patch_size": PATCH_SIZE,
        "occlusion_baseline_oct": "mean intensity of valid scan content",
        "occlusion_baseline_rnflt": "0 in train-normalized RNFLT space",
    })

pd.DataFrame(records).to_csv(
    XAI_DIR / "occlusion_sensitivity_summary.csv",
    index=False,
)

print("Occlusion Sensitivity finished.")
print(f"Exact cases analysed: {len(selected_cases)}")
print(f"Outputs saved to: {XAI_DIR}")

## Harvard-GF MC Dropout uncertainty analysis

In [ ]:
# MC Dropout uncertainty analysis for the existing Harvard-GF model.
# Runs 50 stochastic inference passes per sample; it does not retrain or change the locked decision threshold.

import json
import hashlib
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

# This cell expects the notebook's setup/model/preprocessing cells to have run.
required_globals = [
    "DEVICE", "OUT_DIR", "BEST_PATH", "LOCK_PATH", "make_model",
    "preprocess_batch", "val_frame", "test_frame", "split_cache_paths",
    "cache_is_valid", "build_split_cache", "impute_invalid_rnflt_map",
    "LOCAL_CACHE_ROOT", "DRIVE_CACHE_ROOT", "N_SCANS", "MODEL_SIZE",
    "MODEL_WIDTH", "SCAN_CHUNK_SIZE", "BATCH_SIZE", "RNFLT_STATS",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the notebook setup/model-definition cells first. Missing: "
        + ", ".join(missing_globals)
        + ". Do not rerun the test-evaluation cell if it says the test was already evaluated."
    )

RUN_DIR = Path(OUT_DIR)
VAL_PRED_PATH = RUN_DIR / "final_validation_predictions.csv"
TEST_PRED_PATH = RUN_DIR / "best_model_test_predictions.csv"
LOCK_PATH = Path(LOCK_PATH)
BEST_PATH = Path(BEST_PATH)

for path in (VAL_PRED_PATH, TEST_PRED_PATH, LOCK_PATH, BEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required saved file not found: {path}")

locked_protocol = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
checkpoint_sha = hashlib.sha256(BEST_PATH.read_bytes()).hexdigest()
if locked_protocol.get("best_checkpoint_sha256") != checkpoint_sha:
    raise RuntimeError("The checkpoint hash differs from the locked evaluation protocol.")
threshold = float(locked_protocol["threshold"])

# Use the exact saved checkpoint; this does not perform training or threshold selection.
checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
model = make_model()
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

# Reuse the verified fixed-slice caches; restore them from Drive if this Runtime was reset.
for split, frame in (("validation", val_frame), ("test", test_frame)):
    if not cache_is_valid(LOCAL_CACHE_ROOT, frame, split):
        build_split_cache(frame, split, allow_test=(split == "test"))
    if not cache_is_valid(LOCAL_CACHE_ROOT, frame, split):
        raise RuntimeError(f"The {split} cache is invalid after verification/restoration.")

class UncertaintyCachedGFDataset(Dataset):
    """Read the existing cache with the exact RNFLT imputation/preprocessing rules."""
    def __init__(self, frame, split):
        self.frame = frame.reset_index(drop=True).copy()
        paths = split_cache_paths(LOCAL_CACHE_ROOT, split)
        self.oct = np.load(paths["oct"], mmap_mode="r")
        self.rnflt = np.load(paths["rnflt"], mmap_mode="r")
        self.labels = self.frame["glaucoma_label"].astype(np.float32).to_numpy()
        self.ids = self.frame["sample_id"].astype(str).to_numpy()
        self.imputed_cache = {}
        self.invalid_fraction_cache = {}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        if index not in self.imputed_cache:
            raw = np.array(self.rnflt[index], dtype=np.float32, copy=True)
            repaired, invalid_fraction = impute_invalid_rnflt_map(raw)
            self.imputed_cache[index] = repaired
            self.invalid_fraction_cache[index] = invalid_fraction
        return {
            "oct_uint8": torch.from_numpy(np.array(self.oct[index], copy=True)),
            "rnflt_imputed": torch.from_numpy(
                np.array(self.imputed_cache[index], dtype=np.float32, copy=True)
            ),
            "rnflt_invalid_fraction": torch.tensor(
                self.invalid_fraction_cache[index], dtype=torch.float32
            ),
            "label": torch.tensor(self.labels[index], dtype=torch.float32),
            "sample_id": self.ids[index],
        }

val_mc_dataset = UncertaintyCachedGFDataset(val_frame, "validation")
test_mc_dataset = UncertaintyCachedGFDataset(test_frame, "test")

MC_PASSES = 50
MC_SEED = 20261016
BATCH_SIZE_MC = int(BATCH_SIZE)
NUM_WORKERS_MC = 0
N_BOOTSTRAP = 2000
HIGH_UNCERTAINTY_FRACTION = 0.20
COVERAGE_TARGETS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

# Fixed seed for reproducible stochastic passes on the same software/device setup.
torch.manual_seed(MC_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MC_SEED)

amp_enabled = bool(globals().get("AMP_ENABLED", False)) and DEVICE.type == "cuda"
def autocast_context():
    if amp_enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    return nullcontext()

# MC Dropout: keep the whole model and all normalization layers in eval mode,
# then activate only Dropout modules. DINO's ResNet remains deterministically in eval mode.
model.eval()
dropout_modules = [m for m in model.modules() if isinstance(m, nn.Dropout)]
for module in dropout_modules:
    module.train()
model.dino_encoder.eval()
if not dropout_modules:
    raise RuntimeError("No nn.Dropout modules were found; MC Dropout cannot be run on this model.")
print(f"MC Dropout active in {len(dropout_modules)} Dropout modules; passes={MC_PASSES}.")

@torch.no_grad()
def run_mc_dropout(dataset, split_name):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_MC,
        shuffle=False,
        num_workers=NUM_WORKERS_MC,
        pin_memory=(DEVICE.type == "cuda"),
    )
    sample_ids, labels = [], []
    mc_batch_outputs = []

    for batch in tqdm(loader, desc=f"MC Dropout: {split_name}"):
        scans, rnflt = preprocess_batch(batch, augment=False)
        b, s = scans.shape[:2]
        flat_scans = scans.reshape(b * s, 1, MODEL_SIZE, MODEL_WIDTH)

        # The DINO encoder has no active dropout. Compute its scan features once per batch,
        # then reuse them for all stochastic downstream passes to avoid 50x OCT encoding cost.
        feature_chunks = []
        model.dino_encoder.eval()
        with autocast_context():
            for start in range(0, len(flat_scans), SCAN_CHUNK_SIZE):
                feature_chunks.append(model.dino_encoder(flat_scans[start:start + SCAN_CHUNK_SIZE]))
            oct_features = torch.cat(feature_chunks, dim=0).reshape(b, s, 512)
            rnflt_features = model.rnflt_encoder.features(rnflt)

        positions = torch.linspace(
            -1.0, 1.0, s, device=oct_features.device, dtype=oct_features.dtype
        )
        pos_grid = positions.view(1, s, 1).expand(b, -1, -1)
        score_input = torch.cat([oct_features, pos_grid], dim=-1)

        batch_probabilities = np.empty((MC_PASSES, b), dtype=np.float32)
        for pass_index in range(MC_PASSES):
            # Run the same downstream calculations as CombinedGFGlaucomaModel.forward,
            # allowing only its existing Dropout layers to sample different masks.
            with autocast_context():
                attention_logits = model.scan_attention(score_input).squeeze(-1)
                weights = torch.softmax(attention_logits.float(), dim=1).to(oct_features.dtype)
                pooled = torch.sum(oct_features * weights.unsqueeze(-1), dim=1)
                pooled_position = torch.sum(
                    positions.view(1, -1) * weights, dim=1, keepdim=True
                )
                oct_embedding = model.oct_projection(
                    torch.cat([pooled, pooled_position], dim=1)
                )
                rnflt_embedding = model.rnflt_encoder.projection(rnflt_features)
                fused = model.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
                logits = model.glaucoma_head(fused).squeeze(-1).float()
                probabilities = torch.sigmoid(logits)
            batch_probabilities[pass_index] = probabilities.detach().cpu().numpy()

        mc_batch_outputs.append(batch_probabilities)
        sample_ids.extend([str(x) for x in batch["sample_id"]])
        labels.extend(batch["label"].cpu().numpy().astype(int).tolist())

    # Shape is [MC_PASSES, number_of_samples].
    probabilities = np.concatenate(mc_batch_outputs, axis=1)
    if probabilities.shape != (MC_PASSES, len(dataset)):
        raise RuntimeError(
            f"Unexpected MC probability matrix shape for {split_name}: {probabilities.shape}"
        )
    if not np.isfinite(probabilities).all():
        raise FloatingPointError(f"Non-finite MC probabilities in {split_name}.")

    eps = 1e-7
    p_clipped = np.clip(probabilities.astype(np.float64), eps, 1.0 - eps)
    mean_p = p_clipped.mean(axis=0)
    predictive_entropy = -(
        mean_p * np.log(mean_p) + (1.0 - mean_p) * np.log(1.0 - mean_p)
    )
    sample_entropy = -(
        p_clipped * np.log(p_clipped)
        + (1.0 - p_clipped) * np.log(1.0 - p_clipped)
    )
    expected_entropy = sample_entropy.mean(axis=0)
    mutual_information = np.maximum(predictive_entropy - expected_entropy, 0.0)
    predictive_variance = p_clipped.var(axis=0, ddof=0)

    return pd.DataFrame({
        "sample_id": sample_ids,
        "y_true_mc": np.asarray(labels, dtype=int),
        "mc_mean_probability": mean_p,
        "mc_probability_std": p_clipped.std(axis=0, ddof=0),
        "mc_predictive_entropy": predictive_entropy,
        "mc_mutual_information": mutual_information,
        "mc_predictive_variance": predictive_variance,
    })


def normalize_sample_id(value):
    text = str(value).strip()
    try:
        numeric = float(text)
        if np.isfinite(numeric) and numeric.is_integer():
            return str(int(numeric))
    except (ValueError, TypeError):
        pass
    return text


def binary_entropy(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1.0 - 1e-7)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))


def merge_with_deterministic_predictions(mc_frame, pred_path, split_name):
    det = pd.read_csv(pred_path, dtype={"sample_id": str})
    if not {"sample_id", "y_true", "probability"}.issubset(det.columns):
        raise RuntimeError(f"Unexpected columns in {pred_path}.")
    det = det[["sample_id", "y_true", "probability"]].copy()
    det["sample_id"] = det["sample_id"].map(normalize_sample_id)
    mc = mc_frame.copy()
    mc["sample_id"] = mc["sample_id"].map(normalize_sample_id)

    if det["sample_id"].duplicated().any() or mc["sample_id"].duplicated().any():
        raise RuntimeError(f"Duplicate sample IDs found in {split_name} predictions.")
    if set(det["sample_id"]) != set(mc["sample_id"]):
        raise RuntimeError(f"Saved and MC sample IDs do not match for {split_name}.")

    merged = det.merge(mc, on="sample_id", how="left", validate="one_to_one", sort=False)
    merged["y_true"] = pd.to_numeric(merged["y_true"], errors="raise").astype(int)
    if not np.array_equal(merged["y_true"].to_numpy(), merged["y_true_mc"].to_numpy(dtype=int)):
        raise RuntimeError(f"Labels differ between saved predictions and {split_name} dataset.")
    merged["det_probability"] = pd.to_numeric(merged["probability"], errors="raise").astype(float)
    if not np.isfinite(merged["det_probability"]).all():
        raise RuntimeError(f"Non-finite deterministic predictions in {split_name}.")

    # Define errors using the original saved deterministic prediction and the locked threshold.
    merged["det_prediction"] = (merged["det_probability"] >= threshold).astype(int)
    merged["error"] = (merged["det_prediction"].to_numpy() != merged["y_true"].to_numpy()).astype(int)

    # Simpler uncertainty baselines for comparison with MC Dropout.
    merged["deterministic_entropy"] = binary_entropy(merged["det_probability"].to_numpy())
    merged["threshold_closeness"] = 1.0 - np.abs(
        merged["det_probability"].to_numpy() - threshold
    )
    merged["mc_mean_minus_deterministic"] = (
        merged["mc_mean_probability"] - merged["det_probability"]
    )
    return merged

# Execute MC inference on validation and test. The validation set sets the uncertainty cutoff;
# test is used only for final post-hoc uncertainty evaluation, never for cutoff selection.
try:
    val_mc_raw = run_mc_dropout(val_mc_dataset, "validation")
    test_mc_raw = run_mc_dropout(test_mc_dataset, "test")
finally:
    # Leave the model in ordinary deterministic evaluation mode after the MC passes.
    model.eval()

val_uncertainty = merge_with_deterministic_predictions(
    val_mc_raw, VAL_PRED_PATH, "validation"
)
test_uncertainty = merge_with_deterministic_predictions(
    test_mc_raw, TEST_PRED_PATH, "test"
)

SCORE_COLUMNS = {
    "MC predictive entropy": "mc_predictive_entropy",
    "MC mutual information": "mc_mutual_information",
    "MC predictive variance": "mc_predictive_variance",
    "Deterministic predictive entropy": "deterministic_entropy",
    "Distance-to-threshold baseline": "threshold_closeness",
}

# Experiment 1: how well does each uncertainty score rank actual deterministic-model errors?
def error_detection_metrics(frame, split_name):
    y_error = frame["error"].to_numpy(dtype=int)
    if len(np.unique(y_error)) < 2:
        raise RuntimeError(f"Both correct and incorrect predictions are required in {split_name}.")
    rows = []
    error_prevalence = float(y_error.mean())
    for score_name, score_col in SCORE_COLUMNS.items():
        scores = frame[score_col].to_numpy(dtype=float)
        rows.append({
            "split": split_name,
            "uncertainty_score": score_name,
            "error_detection_auroc": float(roc_auc_score(y_error, scores)),
            "error_detection_average_precision": float(average_precision_score(y_error, scores)),
            "error_rate_baseline_for_AUPRC": error_prevalence,
            "n": int(len(frame)),
            "errors": int(y_error.sum()),
        })
    return rows

error_detection_table = pd.DataFrame(
    error_detection_metrics(val_uncertainty, "validation")
    + error_detection_metrics(test_uncertainty, "test")
)

# Experiment 2: risk-coverage. Keep the least-uncertain samples at each coverage level.
risk_rows = []
for score_name, score_col in SCORE_COLUMNS.items():
    scores = test_uncertainty[score_col].to_numpy(dtype=float)
    errors = test_uncertainty["error"].to_numpy(dtype=int)
    order = np.argsort(scores, kind="mergesort")  # Low uncertainty first.
    per_score = []
    for target_coverage in COVERAGE_TARGETS:
        n_keep = max(1, int(np.ceil(target_coverage * len(test_uncertainty))))
        kept_idx = order[:n_keep]
        risk = float(errors[kept_idx].mean())
        row = {
            "uncertainty_score": score_name,
            "target_coverage": target_coverage,
            "actual_coverage": n_keep / len(test_uncertainty),
            "n_kept": int(n_keep),
            "errors_kept": int(errors[kept_idx].sum()),
            "selective_error_rate": risk,
        }
        risk_rows.append(row)
        per_score.append(row)
    auc_rows = sorted(per_score, key=lambda r: r["actual_coverage"])
    cov = np.asarray([r["actual_coverage"] for r in auc_rows], dtype=float)
    risk = np.asarray([r["selective_error_rate"] for r in auc_rows], dtype=float)
    aurc = float(np.sum((risk[1:] + risk[:-1]) * 0.5 * np.diff(cov)))
    for row in per_score:
        row["discrete_AURC_for_score"] = aurc

risk_coverage_table = pd.DataFrame(risk_rows)

# Experiment 3: define the top-20%-uncertainty cutoff from validation, then freeze it on test.
def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return (float("nan"), float("nan"))
    p = successes / total
    den = 1.0 + z * z / total
    center = (p + z * z / (2.0 * total)) / den
    half = z * np.sqrt(p * (1.0 - p) / total + z * z / (4.0 * total * total)) / den
    return (float(max(0.0, center - half)), float(min(1.0, center + half)))

rng = np.random.default_rng(MC_SEED)
group_rows = []
for score_index, (score_name, score_col) in enumerate(SCORE_COLUMNS.items()):
    val_scores = val_uncertainty[score_col].to_numpy(dtype=float)
    test_scores = test_uncertainty[score_col].to_numpy(dtype=float)
    # The threshold is estimated only from validation uncertainty scores.
    cutoff = float(np.quantile(val_scores, 1.0 - HIGH_UNCERTAINTY_FRACTION, method="linear"))
    val_high = val_scores >= cutoff
    test_high = test_scores >= cutoff
    test_errors = test_uncertainty["error"].to_numpy(dtype=int)
    high_errors = test_errors[test_high]
    low_errors = test_errors[~test_high]

    if len(high_errors) == 0 or len(low_errors) == 0:
        high_rate = float("nan") if len(high_errors) == 0 else float(high_errors.mean())
        low_rate = float("nan") if len(low_errors) == 0 else float(low_errors.mean())
        high_ci = (float("nan"), float("nan")) if len(high_errors) == 0 else wilson_interval(int(high_errors.sum()), len(high_errors))
        low_ci = (float("nan"), float("nan")) if len(low_errors) == 0 else wilson_interval(int(low_errors.sum()), len(low_errors))
        risk_diff = float("nan")
        risk_diff_ci = (float("nan"), float("nan"))
    else:
        high_rate = float(high_errors.mean())
        low_rate = float(low_errors.mean())
        high_ci = wilson_interval(int(high_errors.sum()), len(high_errors))
        low_ci = wilson_interval(int(low_errors.sum()), len(low_errors))
        risk_diff = high_rate - low_rate
        # Parametric bootstrap for the difference in two independent Bernoulli error rates.
        boot_high = rng.binomial(len(high_errors), high_rate, size=N_BOOTSTRAP) / len(high_errors)
        boot_low = rng.binomial(len(low_errors), low_rate, size=N_BOOTSTRAP) / len(low_errors)
        risk_diff_ci = tuple(np.percentile(boot_high - boot_low, [2.5, 97.5]).astype(float))

    group_rows.append({
        "uncertainty_score": score_name,
        "validation_cutoff_80th_percentile": cutoff,
        "validation_high_group_fraction_actual": float(val_high.mean()),
        "test_high_group_fraction_actual": float(test_high.mean()),
        "test_high_n": int(test_high.sum()),
        "test_low_n": int((~test_high).sum()),
        "high_uncertainty_error_rate": high_rate,
        "high_error_ci95_low": high_ci[0],
        "high_error_ci95_high": high_ci[1],
        "low_uncertainty_error_rate": low_rate,
        "low_error_ci95_low": low_ci[0],
        "low_error_ci95_high": low_ci[1],
        "high_minus_low_error_rate": risk_diff,
        "difference_ci95_low": risk_diff_ci[0],
        "difference_ci95_high": risk_diff_ci[1],
        "validation_cutoff_note": "Threshold fixed from validation; tied values can make actual group fraction differ from 20%.",
    })

group_comparison_table = pd.DataFrame(group_rows)

# Save per-sample results and all requested evaluation tables.
UNCERTAINTY_DIR = RUN_DIR / "mc_dropout_uncertainty_analysis"
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)

val_uncertainty.to_csv(UNCERTAINTY_DIR / "validation_uncertainty_per_sample.csv", index=False)
test_uncertainty.to_csv(UNCERTAINTY_DIR / "test_uncertainty_per_sample.csv", index=False)
error_detection_table.to_csv(UNCERTAINTY_DIR / "error_detection_metrics.csv", index=False)
risk_coverage_table.to_csv(UNCERTAINTY_DIR / "risk_coverage_test.csv", index=False)
group_comparison_table.to_csv(UNCERTAINTY_DIR / "high_vs_low_uncertainty_test.csv", index=False)

# Risk-coverage plot for the three MC Dropout scores and two simple baselines.
fig, ax = plt.subplots(figsize=(8, 6))
for score_name in SCORE_COLUMNS:
    subset = risk_coverage_table[risk_coverage_table["uncertainty_score"] == score_name]
    subset = subset.sort_values("actual_coverage")
    ax.plot(
        subset["actual_coverage"], subset["selective_error_rate"],
        marker="o", linewidth=1.8, label=score_name
    )
ax.set(
    xlim=(0, 1), ylim=(0, 1),
    xlabel="Coverage (fraction of least-uncertain samples retained)",
    ylabel="Error rate among retained samples",
    title="Risk–Coverage Curve on Test",
)
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(UNCERTAINTY_DIR / "risk_coverage_test.png", dpi=300, bbox_inches="tight")
fig.savefig(UNCERTAINTY_DIR / "risk_coverage_test.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# Plot the test error-detection AUROC for MC Dropout and simple baselines.
test_rank_metrics = error_detection_table[error_detection_table["split"] == "test"].copy()
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(test_rank_metrics["uncertainty_score"], test_rank_metrics["error_detection_auroc"])
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.2, label="Chance (AUROC=0.5)")
ax.set(
    ylim=(0, 1),
    ylabel="AUROC for ranking model errors",
    title="Error-detection AUROC on Test",
)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(UNCERTAINTY_DIR / "error_detection_auroc_test.png", dpi=300, bbox_inches="tight")
fig.savefig(UNCERTAINTY_DIR / "error_detection_auroc_test.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

run_summary = {
    "status": "MC_DROPOUT_UNCERTAINTY_ANALYSIS_COMPLETE",
    "dataset": "harvard_gf",
    "checkpoint_sha256": checkpoint_sha,
    "locked_decision_threshold": threshold,
    "decision_errors_defined_using": "saved deterministic probability >= locked validation threshold; no decision threshold was changed",
    "mc_dropout_passes": MC_PASSES,
    "active_dropout_module_count": len(dropout_modules),
    "uncertainty_scores": list(SCORE_COLUMNS.keys()),
    "baseline_error_rate_test": float(test_uncertainty["error"].mean()),
    "high_uncertainty_group_fraction_target": HIGH_UNCERTAINTY_FRACTION,
    "high_uncertainty_cutoffs_selected_on": "validation only, separately for each uncertainty score",
    "test_used_for_model_or_threshold_selection": False,
    "bootstrap_replicates_for_group_difference_ci": N_BOOTSTRAP,
    "seed": MC_SEED,
    "output_directory": str(UNCERTAINTY_DIR),
}
(UNCERTAINTY_DIR / "run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

print("\nExperiment 1 and 4 — error-ranking metrics (higher AUROC/AUPRC is better):")
try:
    display(error_detection_table.round(4))
except NameError:
    print(error_detection_table.round(4).to_string(index=False))

print("\nExperiment 3 — test error rate in validation-defined high/low uncertainty groups:")
try:
    display(group_comparison_table.round(4))
except NameError:
    print(group_comparison_table.round(4).to_string(index=False))

print("\nExperiment 2 — risk-coverage rows:")
try:
    display(risk_coverage_table.round(4))
except NameError:
    print(risk_coverage_table.round(4).to_string(index=False))

print("\nSaved uncertainty analysis outputs to:", UNCERTAINTY_DIR)
print(json.dumps(run_summary, indent=2))

## Part B — Harvard-GDP external evaluation

The final GF model is evaluated on the independent Harvard-GDP glaucoma test set without retraining or fine-tuning. The same locked threshold **0.404529** is used.

In [ ]:
from google.colab import drive
from pathlib import Path
from contextlib import nullcontext
import os, io, json, zipfile, hashlib, math, gc, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, matthews_corrcoef, brier_score_loss,
    log_loss, roc_curve, precision_recall_curve,
)

drive.mount('/content/drive')
MYDRIVE = Path('/content/drive/MyDrive')
PROJECT_CANDIDATES = [
    MYDRIVE / 'backup' / 'glaucoma_oct_research',
    MYDRIVE / 'glaucoma_oct_research',
]

def project_ready(root):
    required = [
        root / 'results/stage10_protocol/main_experiment_manifest.csv',
        root / 'results/stage10_protocol/ssl_training_manifest.csv',
        root / 'results/stage10_protocol/leakage_audit.csv',
        root / 'checkpoints/stage11_ssl_dino_v2_1/dino_teacher_resnet18_encoder_axis0_ssl_v2.pt',
        root / 'checkpoints/stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch/best_model.pt',
        root / 'results/stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch/evaluation_protocol_locked.json',
        root / 'results/stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch/training_config.json',
        root / 'results/stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch/rnflt_stats_gf_train_only.json',
    ]
    return all(p.is_file() for p in required)

PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if project_ready(p)), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find the Stage 10 manifests, DINO checkpoint, GF best_model.pt, and locked GF protocol '
        'in either MyDrive/glaucoma_oct_research or MyDrive/backup/glaucoma_oct_research.'
    )

STAGE10_DIR = PROJECT_DIR / 'results/stage10_protocol'
MANIFEST_PATH = STAGE10_DIR / 'main_experiment_manifest.csv'
SSL_MANIFEST_PATH = STAGE10_DIR / 'ssl_training_manifest.csv'
LEAKAGE_AUDIT_PATH = STAGE10_DIR / 'leakage_audit.csv'
DINO_PATH = PROJECT_DIR / 'checkpoints/stage11_ssl_dino_v2_1/dino_teacher_resnet18_encoder_axis0_ssl_v2.pt'
RUN_TAG = 'stage13_harvard_gf_glaucoma_only_v2_rnflt_imputed_1ch'
MODEL_OUT_DIR = PROJECT_DIR / 'results' / RUN_TAG
BEST_PATH = PROJECT_DIR / 'checkpoints' / RUN_TAG / 'best_model.pt'
CONFIG_PATH = MODEL_OUT_DIR / 'training_config.json'
STATS_PATH = MODEL_OUT_DIR / 'rnflt_stats_gf_train_only.json'
LOCK_PATH = MODEL_OUT_DIR / 'evaluation_protocol_locked.json'

OUT_DIR = PROJECT_DIR / 'results/stage14_gdp_glaucoma_external_eval_gf_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
N_SCANS = 12
MODEL_SIZE = 224
MODEL_WIDTH = 320
SCAN_CHUNK_SIZE = 12
BATCH_SIZE = 4
NUM_WORKERS = 0
MC_PASSES = 50
MC_SEED = 20261019
N_BOOTSTRAP = 2000
N_CALIBRATION_BINS = 5
COVERAGE_TARGETS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
HIGH_UNCERTAINTY_FRACTION = 0.20


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

print('Project root:', PROJECT_DIR)
print('Device:', DEVICE)
print('GF checkpoint:', BEST_PATH)
print('Output directory:', OUT_DIR)
for path in [MANIFEST_PATH, SSL_MANIFEST_PATH, LEAKAGE_AUDIT_PATH, DINO_PATH, BEST_PATH, CONFIG_PATH, STATS_PATH, LOCK_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Required project file missing: {path}')

CHECKPOINT_SHA256 = sha256_file(BEST_PATH)
DINO_SHA256 = sha256_file(DINO_PATH)
locked_protocol = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
if locked_protocol.get('status') != 'LOCKED_BEFORE_TEST':
    raise RuntimeError(f'GF evaluation threshold is not marked as locked: {locked_protocol.get("status")}')
if locked_protocol.get('best_checkpoint_sha256') != CHECKPOINT_SHA256:
    raise RuntimeError('GF checkpoint SHA-256 does not match the locked evaluation protocol.')
#THRESHOLD = float(locked_protocol['threshold'])
THRESHOLD=0.404529
if not np.isfinite(THRESHOLD) or not (0.0 <= THRESHOLD <= 1.0):
    raise RuntimeError(f'Invalid threshold stored in GF lock file: {THRESHOLD}')

training_config_json = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
if 'config_hash' in locked_protocol and training_config_json.get('config_hash') != locked_protocol.get('config_hash'):
    raise RuntimeError('GF training config hash does not match the locked protocol.')

print('Locked GF-validation threshold:', THRESHOLD)
print('Threshold method:', locked_protocol.get('threshold_selection', 'recorded in locked protocol'))
print('GF checkpoint SHA-256:', CHECKPOINT_SHA256)
print('DINO checkpoint SHA-256:', DINO_SHA256)


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={'sample_id': str})
ssl_manifest = pd.read_csv(SSL_MANIFEST_PATH, dtype={'sample_id': str})
leakage_audit = pd.read_csv(LEAKAGE_AUDIT_PATH)

required_manifest_columns = {
    'dataset', 'sample_id', 'subject_key', 'protocol_split', 'glaucoma_split',
    'glaucoma_label', 'oct_locator', 'rnflt_locator'
}
missing = sorted(required_manifest_columns - set(manifest.columns))
if missing:
    raise RuntimeError(f'Main manifest is missing required columns: {missing}')
if not {'dataset', 'sample_id', 'source_split', 'ssl_role'}.issubset(ssl_manifest.columns):
    raise RuntimeError('SSL manifest is missing dataset/sample_id/source_split/ssl_role.')
if leakage_audit.empty or leakage_audit['status'].astype(str).str.upper().ne('PASS').any():
    raise RuntimeError('The saved Stage 10 leakage audit is not fully PASS.')

for frame in (manifest, ssl_manifest):
    frame['dataset'] = frame['dataset'].astype(str).str.strip().str.lower()
    frame['sample_id'] = frame['sample_id'].astype(str).str.strip()
    if 'subject_key' not in frame.columns:
        frame['subject_key'] = frame['dataset'] + '::' + frame['sample_id']
    else:
        frame['subject_key'] = frame['subject_key'].astype(str).str.strip()

manifest['protocol_split'] = manifest['protocol_split'].astype(str).str.strip().str.lower()
manifest['glaucoma_split'] = manifest['glaucoma_split'].astype(str).str.strip().str.lower()
manifest['glaucoma_label'] = pd.to_numeric(manifest['glaucoma_label'], errors='coerce')
ssl_manifest['source_split'] = ssl_manifest['source_split'].astype(str).str.strip().str.lower()
ssl_manifest['ssl_role'] = ssl_manifest['ssl_role'].astype(str).str.strip().str.lower()

if len(ssl_manifest) != 2600:
    raise RuntimeError(f'Expected 2600 total DINO SSL manifest rows; found {len(ssl_manifest)}.')
GF_SSL = ssl_manifest.loc[ssl_manifest['dataset'].eq('harvard_gf')].copy()
GDP_SSL = ssl_manifest.loc[ssl_manifest['dataset'].eq('harvard_gdp')].copy()
if len(GF_SSL) != 2100:
    raise RuntimeError(f'Expected 2100 GF-train SSL rows; found {len(GF_SSL)}.')
if not GF_SSL['ssl_role'].eq('gf_train').all():
    raise RuntimeError('GF SSL rows have unexpected ssl_role values.')
if not GF_SSL['source_split'].eq('train').all():
    raise RuntimeError('GF SSL rows must originate from the GF train split.')
GDP_TEST = manifest.loc[
    manifest['dataset'].eq('harvard_gdp') & manifest['glaucoma_split'].eq('test')
].copy()
GDP_TEST = GDP_TEST.sort_values('sample_id', kind='stable').reset_index(drop=True)

if len(GDP_SSL) != 500:
    raise RuntimeError(f'Expected the frozen 500-sample GDP SSL pool; found {len(GDP_SSL)}.')
if not GDP_SSL['source_split'].eq('ssl_pool').all():
    raise RuntimeError('GDP SSL rows do not all have source_split=ssl_pool.')
if not GDP_SSL['ssl_role'].eq('gdp_ssl_pool').all():
    raise RuntimeError('GDP SSL rows have unexpected ssl_role values.')
if len(GDP_TEST) != 75:
    raise RuntimeError(f'Expected 75 GDP glaucoma-test samples in the frozen protocol; found {len(GDP_TEST)}.')
if GDP_TEST['sample_id'].duplicated().any() or GDP_TEST['subject_key'].duplicated().any():
    raise RuntimeError('Duplicate GDP test identifiers detected.')
if not GDP_TEST['protocol_split'].eq('test').all():
    raise RuntimeError('Some selected glaucoma-test rows do not belong to the frozen progression test group.')
if GDP_TEST['glaucoma_label'].isna().any() or not set(GDP_TEST['glaucoma_label'].astype(int).unique()).issubset({0, 1}):
    raise RuntimeError('GDP test glaucoma labels must be complete binary values (0/1).')
if GDP_TEST['oct_locator'].isna().any() or GDP_TEST['rnflt_locator'].isna().any():
    raise RuntimeError('GDP test manifest contains missing OCT/RNFLT locators.')
if GDP_TEST['oct_locator'].astype(str).str.strip().eq('').any() or GDP_TEST['rnflt_locator'].astype(str).str.strip().eq('').any():
    raise RuntimeError('GDP test manifest contains blank OCT/RNFLT locators.')

ssl_sample_ids = set(GDP_SSL['sample_id'].astype(str))
ssl_subject_keys = set(GDP_SSL['subject_key'].astype(str))
test_sample_ids = set(GDP_TEST['sample_id'].astype(str))
test_subject_keys = set(GDP_TEST['subject_key'].astype(str))
overlap_samples = test_sample_ids & ssl_sample_ids
overlap_subjects = test_subject_keys & ssl_subject_keys
if overlap_samples or overlap_subjects:
    raise RuntimeError(
        'LEAKAGE GUARD FAILED: GDP test overlaps the SSL pool. '
        f'sample_id overlap={len(overlap_samples)}, subject_key overlap={len(overlap_subjects)}. '
        'No test images have been opened.'
    )

ssl_audit = {
    'status': 'PASS',
    'model_supervised_training_dataset': 'harvard_gf',
    'model_self_supervised_exposure': '2100 GF train + 500 GDP ssl_pool, unlabeled',
    'selected_dataset': 'harvard_gdp',
    'selected_split': 'glaucoma_split=test',
    'test_count': int(len(GDP_TEST)),
    'total_ssl_manifest_count': int(len(ssl_manifest)),
    'gf_ssl_train_count': int(len(GF_SSL)),
    'gdp_ssl_pool_count': int(len(GDP_SSL)),
    'test_ssl_sample_id_overlap': int(len(overlap_samples)),
    'test_ssl_subject_key_overlap': int(len(overlap_subjects)),
    'test_samples_loaded_in_this_cell': False,
    'progression_labels_used': False,
}
(OUT_DIR / 'test_ssl_exposure_audit.json').write_text(json.dumps(ssl_audit, indent=2), encoding='utf-8')
print('SSL/test isolation audit: PASS')
print('GDP test sample count:', len(GDP_TEST))
print('No GDP test IDs overlap the 500-sample SSL pool.')
print('No images have been opened in this cell.')


In [ ]:
def read_locator_bytes(locator):
    locator = str(locator).strip()
    if locator.startswith('zip://'):
        archive, member = locator[len('zip://'):].split('!', 1)
        if not Path(archive).is_file():
            raise FileNotFoundError(f'Archive referenced by the manifest is missing: {archive}. Restore/download the raw dataset before evaluation.')
        with zipfile.ZipFile(archive, 'r') as zf:
            return zf.read(member)
    if locator.startswith('nestedzip://'):
        outer_path, nested_name, member = locator[len('nestedzip://'):].split('!', 2)
        if not Path(outer_path).is_file():
            raise FileNotFoundError(f'Outer archive referenced by the manifest is missing: {outer_path}. Restore/download the raw dataset before evaluation.')
        with zipfile.ZipFile(outer_path, 'r') as outer:
            nested_bytes = outer.read(nested_name)
        with zipfile.ZipFile(io.BytesIO(nested_bytes), 'r') as nested:
            return nested.read(member)
    if locator.startswith('file://'):
        locator = locator[len('file://'):]
    path = Path(locator)
    if not path.is_file():
        raise FileNotFoundError(
            f'Raw dataset file referenced by manifest is unavailable: {path}\n'
            'The backup copy intentionally excludes raw datasets. Restore/download the relevant raw GDP files '
            'and make sure the manifest locator points to their current location.'
        )
    return path.read_bytes()


def read_npz_fields(locator, fields):
    raw = read_locator_bytes(locator)
    try:
        with np.load(io.BytesIO(raw), allow_pickle=False) as data:
            return {key: np.array(data[key], copy=True) for key in fields if key in data.files}
    finally:
        del raw


def choose_bscan_indices(depth, n_scans=N_SCANS):
    edges = np.linspace(0, depth, n_scans + 1, dtype=np.int32)
    indices = ((edges[:-1] + edges[1:] - 1) // 2).astype(np.int64)
    if len(indices) != n_scans or len(np.unique(indices)) != n_scans or np.any(indices < 0) or np.any(indices >= depth):
        raise RuntimeError('Invalid fixed B-scan index selection.')
    return indices

SCAN_INDICES = choose_bscan_indices(200)
print('Fixed axis-0 scan indices:', SCAN_INDICES.tolist())


def load_input_arrays(frame, dataset_name):
    frame = frame.reset_index(drop=True).copy()
    if frame.empty:
        raise ValueError(f'No samples supplied for {dataset_name}.')
    n = len(frame)
    oct_key = 'bscans' if dataset_name == 'harvard_gdp' else 'oct_bscans'
    oct_array = None
    rnflt_array = None
    fixed_oct_shape = None
    fixed_rnflt_shape = None

    for i, row in enumerate(tqdm(frame.itertuples(index=False), total=n, desc=f'Load {dataset_name} inputs')):
        row_dict = row._asdict()
        fields = read_npz_fields(row_dict['oct_locator'], {oct_key, 'bscans', 'oct_bscans', 'rnflt'})
        volume = fields.get(oct_key)
        if volume is None:
            alternate = 'oct_bscans' if oct_key == 'bscans' else 'bscans'
            volume = fields.get(alternate)
        if volume is None:
            raise KeyError(f'Missing OCT volume field ({oct_key}) in {row_dict["oct_locator"]}.')
        if volume.dtype != np.uint8 or volume.ndim != 3 or volume.shape[0] != 200 or volume.shape[1] != 200:
            raise ValueError(
                f'Unexpected OCT shape/dtype for {row_dict["sample_id"]}: '
                f'{volume.shape}, {volume.dtype}; expected uint8 [200, 200, W].'
            )
        if dataset_name == 'harvard_gdp' and volume.shape[2] != 300:
            raise ValueError(f'Unexpected GDP B-scan width for {row_dict["sample_id"]}: {volume.shape}.')
        if dataset_name == 'harvard_gf' and volume.shape[2] != 200:
            raise ValueError(f'Unexpected GF B-scan width for {row_dict["sample_id"]}: {volume.shape}.')

        selected = np.asarray(volume[SCAN_INDICES, :, :], dtype=np.uint8)
        current_oct_shape = tuple(selected.shape)
        if oct_array is None:
            fixed_oct_shape = current_oct_shape[1:]
            oct_array = np.empty((n, N_SCANS, *fixed_oct_shape), dtype=np.uint8)
        elif current_oct_shape[1:] != fixed_oct_shape:
            raise ValueError(f'Inconsistent B-scan shape in {dataset_name}: {current_oct_shape[1:]} vs {fixed_oct_shape}.')
        oct_array[i] = selected

        rnflt = fields.get('rnflt')
        del volume, selected, fields
        if rnflt is None:
            rnflt = read_npz_fields(row_dict['rnflt_locator'], {'rnflt'}).get('rnflt')
        if rnflt is None:
            raise KeyError(f'Missing RNFLT map for {row_dict["sample_id"]}.')
        rnflt = np.asarray(rnflt, dtype=np.float32).squeeze()
        if rnflt.ndim != 2:
            raise ValueError(f'Expected 2D RNFLT map for {row_dict["sample_id"]}; got {rnflt.shape}.')
        current_rnflt_shape = tuple(rnflt.shape)
        if rnflt_array is None:
            fixed_rnflt_shape = current_rnflt_shape
            rnflt_array = np.empty((n, *fixed_rnflt_shape), dtype=np.float32)
        elif current_rnflt_shape != fixed_rnflt_shape:
            raise ValueError(f'Inconsistent RNFLT shapes in {dataset_name}: {current_rnflt_shape} vs {fixed_rnflt_shape}.')
        rnflt_array[i] = rnflt
        del rnflt

    labels = pd.to_numeric(frame['glaucoma_label'], errors='raise').to_numpy(dtype=np.int64)
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError('Glaucoma labels must be binary 0/1.')
    return {
        'oct': oct_array,
        'rnflt': rnflt_array,
        'labels': labels,
        'sample_ids': frame['sample_id'].astype(str).to_numpy(),
        'subject_keys': frame['subject_key'].astype(str).to_numpy(),
        'frame': frame,
    }


def impute_invalid_rnflt_map(raw_map):
    values = np.asarray(raw_map, dtype=np.float32).squeeze()
    if values.ndim != 2:
        raise ValueError(f'Expected a 2D RNFLT map; got {values.shape}.')
    valid = np.isfinite(values) & (values >= 0)
    invalid_fraction = float(1.0 - valid.mean())
    if not valid.any():
        raise RuntimeError('RNFLT map contains no finite non-negative pixels.')
    if valid.all():
        return values.copy(), invalid_fraction
    filled = values.copy()
    nearest_indices = distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled[~valid] = values[tuple(nearest_indices[:, ~valid])]
    if not np.isfinite(filled).all() or np.any(filled < 0):
        raise RuntimeError('RNFLT imputation did not produce finite non-negative values.')
    return filled, invalid_fraction


class ArrayGlaucomaDataset(Dataset):
    def __init__(self, arrays):
        self.arrays = arrays
        self.frame = arrays['frame'].reset_index(drop=True).copy()
        self._rnflt_cache = {}
        self._invalid_fraction = {}
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        if index not in self._rnflt_cache:
            repaired, frac = impute_invalid_rnflt_map(self.arrays['rnflt'][index])
            self._rnflt_cache[index] = repaired
            self._invalid_fraction[index] = frac
        return {
            'oct_uint8': torch.from_numpy(np.array(self.arrays['oct'][index], copy=True)),
            'rnflt_imputed': torch.from_numpy(np.array(self._rnflt_cache[index], dtype=np.float32, copy=True)),
            'rnflt_invalid_fraction': torch.tensor(self._invalid_fraction[index], dtype=torch.float32),
            'label': torch.tensor(float(self.arrays['labels'][index]), dtype=torch.float32),
            'sample_id': str(self.arrays['sample_ids'][index]),
            'subject_key': str(self.arrays['subject_keys'][index]),
        }


def preprocess_batch(batch, augment=False):
    oct_uint8 = batch['oct_uint8'].to(DEVICE, non_blocking=True)
    rnflt_imputed = batch['rnflt_imputed'].to(DEVICE, non_blocking=True).float()
    b, s, h, w = oct_uint8.shape
    scans = oct_uint8.reshape(b * s, 1, h, w).float().div_(255.0)
    resized = F.interpolate(scans, size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False)
    canvas = torch.zeros((b * s, 1, MODEL_SIZE, MODEL_WIDTH), dtype=resized.dtype, device=resized.device)
    left = (MODEL_WIDTH - MODEL_SIZE) // 2
    canvas[:, :, :, left:left + MODEL_SIZE] = resized
    scans = canvas.reshape(b, s, 1, MODEL_SIZE, MODEL_WIDTH)
    if augment:
        raise RuntimeError('Augmentation is disabled during external evaluation.')
    rnflt = F.interpolate(rnflt_imputed.unsqueeze(1), size=(MODEL_SIZE, MODEL_SIZE), mode='bilinear', align_corners=False)
    rnflt = ((rnflt - float(RNFLT_STATS['mean'])) / float(RNFLT_STATS['std'])).clamp(-5, 5)
    return scans, rnflt

GDP_TEST_ARRAYS = load_input_arrays(GDP_TEST, 'harvard_gdp')
test_dataset = ArrayGlaucomaDataset(GDP_TEST_ARRAYS)
print('GDP test OCT array:', GDP_TEST_ARRAYS['oct'].shape, GDP_TEST_ARRAYS['oct'].dtype)
print('GDP test raw RNFLT array:', GDP_TEST_ARRAYS['rnflt'].shape, GDP_TEST_ARRAYS['rnflt'].dtype)
print('Input arrays are held in Colab RAM only; no large cache was written to Drive.')


In [ ]:
def autocast_context():
    if DEVICE.type == 'cuda':
        return torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True)
    return nullcontext()


def build_dino_encoder():
    encoder = models.resnet18(weights=None)
    encoder.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    encoder.fc = nn.Identity()
    dino_checkpoint = torch.load(DINO_PATH, map_location='cpu', weights_only=False)
    if dino_checkpoint.get('training_version') != 'dino_v2_1_parametrized_weight_norm_separate_teacher_fp32_loss':
        raise RuntimeError(f'Unexpected DINO training version: {dino_checkpoint.get("training_version")!r}.')
    if dino_checkpoint.get('architecture') != 'torchvision_resnet18_grayscale' or int(dino_checkpoint.get('input_channels', -1)) != 1:
        raise RuntimeError('DINO architecture/channel metadata does not match grayscale ResNet-18.')
    if list(dino_checkpoint.get('input_canvas_hw', [])) != [224, 320]:
        raise RuntimeError('DINO checkpoint canvas is not the expected 224x320.')
    if not str(dino_checkpoint.get('axis_policy', '')).startswith('axis-0'):
        raise RuntimeError('DINO checkpoint does not document native axis-0 ordering.')
    loaded = encoder.load_state_dict(dino_checkpoint['state_dict'], strict=True)
    if loaded.missing_keys or loaded.unexpected_keys:
        raise RuntimeError(f'Strict DINO encoder load failed: {loaded}.')
    return encoder


class Residual2DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.skip = nn.Identity() if in_channels == out_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.GroupNorm(8, out_channels),
        )
    def forward(self, x):
        residual = self.skip(x)
        x = F.gelu(self.norm1(self.conv1(x)))
        return F.gelu(self.norm2(self.conv2(x)) + residual)


class RNFLTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2, bias=False),
            nn.GroupNorm(8, 32), nn.GELU(),
            Residual2DBlock(32, 32),
            Residual2DBlock(32, 64, stride=2),
            Residual2DBlock(64, 128, stride=2),
            Residual2DBlock(128, 128, stride=2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.projection = nn.Sequential(nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10))
    def forward(self, x):
        return self.projection(self.features(x))


class CombinedGFGlaucomaModel(nn.Module):
    def __init__(self, dino_encoder):
        super().__init__()
        self.dino_encoder = dino_encoder
        self.scan_attention = nn.Sequential(nn.Linear(513, 128), nn.Tanh(), nn.Dropout(0.10), nn.Linear(128, 1))
        self.oct_projection = nn.Sequential(nn.Linear(513, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10))
        self.rnflt_encoder = RNFLTEncoder()
        self.fusion = nn.Sequential(
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.10),
        )
        self.glaucoma_head = nn.Linear(64, 1)
        for parameter in self.dino_encoder.parameters():
            parameter.requires_grad_(False)
        self.dino_encoder.eval()
    def train(self, mode=True):
        super().train(mode)
        self.dino_encoder.eval()
        return self
    def encode_oct(self, scans):
        if scans.ndim != 5 or tuple(scans.shape[1:]) != (N_SCANS, 1, MODEL_SIZE, MODEL_WIDTH):
            raise ValueError(f'Expected OCT [B,{N_SCANS},1,{MODEL_SIZE},{MODEL_WIDTH}], got {tuple(scans.shape)}.')
        b, s = scans.shape[:2]
        flat = scans.reshape(b * s, 1, MODEL_SIZE, MODEL_WIDTH)
        chunks = []
        self.dino_encoder.eval()
        for start in range(0, len(flat), SCAN_CHUNK_SIZE):
            with torch.no_grad():
                features = self.dino_encoder(flat[start:start + SCAN_CHUNK_SIZE])
            chunks.append(features)
        features = torch.cat(chunks, dim=0).reshape(b, s, 512)
        positions = torch.linspace(-1.0, 1.0, s, device=features.device, dtype=features.dtype)
        pos_grid = positions.view(1, s, 1).expand(b, -1, -1)
        score_input = torch.cat([features, pos_grid], dim=-1)
        weights = torch.softmax(self.scan_attention(score_input).squeeze(-1).float(), dim=1).to(features.dtype)
        pooled = torch.sum(features * weights.unsqueeze(-1), dim=1)
        pooled_position = torch.sum(positions.view(1, -1) * weights, dim=1, keepdim=True)
        return self.oct_projection(torch.cat([pooled, pooled_position], dim=1))
    def forward(self, scans, rnflt):
        oct_embedding = self.encode_oct(scans)
        rnflt_embedding = self.rnflt_encoder(rnflt)
        fused = self.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
        return self.glaucoma_head(fused).squeeze(-1)


config_hash_in_file = training_config_json.get('config_hash')
checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
if checkpoint.get('status') != 'BEST_GF_GLAUCOMA_MODEL':
    raise RuntimeError(f'Unexpected GF checkpoint status: {checkpoint.get("status")!r}.')
if checkpoint.get('config_hash') != config_hash_in_file:
    raise RuntimeError('Checkpoint config_hash differs from training_config.json.')
if checkpoint.get('dino_sha256') != DINO_SHA256:
    raise RuntimeError('DINO checkpoint SHA-256 differs from GF checkpoint provenance.')

config_stats = checkpoint.get('config', {}).get('rnflt_stats', {})
file_stats = json.loads(STATS_PATH.read_text(encoding='utf-8'))
if file_stats.get('source_split') != 'train' or file_stats.get('dataset') != 'harvard_gf':
    raise RuntimeError('RNFLT normalization statistics are not marked as GF-train-only.')
if config_stats and (
    not np.isclose(float(config_stats['mean']), float(file_stats['mean']), rtol=0, atol=1e-8)
    or not np.isclose(float(config_stats['std']), float(file_stats['std']), rtol=0, atol=1e-8)
):
    raise RuntimeError('RNFLT statistics differ between the saved model config and stats file.')
RNFLT_STATS = {'mean': float(file_stats['mean']), 'std': float(file_stats['std'])}
if RNFLT_STATS['std'] <= 0:
    raise RuntimeError('Invalid RNFLT standard deviation.')

model = CombinedGFGlaucomaModel(build_dino_encoder()).to(DEVICE)
load_result = model.load_state_dict(checkpoint['model_state_dict'], strict=True)
if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(f'Strict GF model load failed: {load_result}.')
model.eval()

print('Loaded final GF model, no fine-tuning performed.')
print('Best GF epoch:', checkpoint.get('best_epoch'))
print('GF-train RNFLT mean/std:', RNFLT_STATS)
print('Locked GF threshold:', THRESHOLD)
print('Checkpoint and DINO provenance checks: PASS')


In [ ]:
def preprocess_batch(batch, augment=False):
    oct_uint8 = batch['oct_uint8'].to(DEVICE, non_blocking=True)
    rnflt_imputed = batch['rnflt_imputed'].to(
        DEVICE, non_blocking=True
    ).float()

    b, s, h, w = oct_uint8.shape
    scans = oct_uint8.reshape(b * s, 1, h, w).float() / 255.0

    # Resize every B-scan to a square, then center it on the model canvas.
    resized = F.interpolate(
        scans,
        size=(MODEL_SIZE, MODEL_SIZE),
        mode='bilinear',
        align_corners=False
    )

    canvas = torch.zeros(
        (b * s, 1, MODEL_SIZE, MODEL_WIDTH),
        dtype=resized.dtype,
        device=resized.device
    )

    left = (MODEL_WIDTH - MODEL_SIZE) // 2
    canvas[:, :, :, left:left + MODEL_SIZE] = resized
    scans = canvas.reshape(b, s, 1, MODEL_SIZE, MODEL_WIDTH)

    if augment:
        brightness = torch.empty(
            (b, 1, 1, 1, 1), device=DEVICE
        ).uniform_(0.92, 1.08)

        gamma = torch.empty(
            (b, 1, 1, 1, 1), device=DEVICE
        ).uniform_(0.92, 1.08)

        scans = (scans * brightness).clamp(0, 1).pow(gamma)

    rnflt = F.interpolate(
        rnflt_imputed.unsqueeze(1),
        size=(MODEL_SIZE, MODEL_SIZE),
        mode='bilinear',
        align_corners=False
    )

    rnflt = (
        (rnflt - float(RNFLT_STATS['mean']))
        / float(RNFLT_STATS['std'])
    ).clamp(-5, 5)

    return scans, rnflt


def display_scan_canvas(scan_uint8):
    scan = (
        torch.as_tensor(scan_uint8, device=device)
        .float()
        .div(255.0)
        .reshape(1, 1, *scan_uint8.shape)
    )

    # Restore the original square-resize visualization.
    resized = F.interpolate(
        scan,
        size=(MODEL_SIZE, MODEL_SIZE),
        mode='bilinear',
        align_corners=False
    )

    canvas = torch.zeros(
        (1, 1, MODEL_SIZE, MODEL_WIDTH),
        device=device,
        dtype=resized.dtype
    )

    left_pad = (MODEL_WIDTH - MODEL_SIZE) // 2
    canvas[:, :, :, left_pad:left_pad + MODEL_SIZE] = resized

    return canvas[0, 0].detach().cpu().numpy()


print("Restored square-resize preprocessing and visualization.")

In [ ]:
def predict_dataset(model, dataset, split_name):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type == 'cuda'))
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Predict {split_name}'):
            scans, rnflt = preprocess_batch(batch, augment=False)
            with autocast_context():
                logits = model(scans, rnflt)
            logits = logits.float()
            probabilities = torch.sigmoid(logits)
            for sid, skey, label, logit, prob, bad_frac in zip(
                batch['sample_id'], batch['subject_key'], batch['label'].numpy(),
                logits.detach().cpu().numpy(), probabilities.detach().cpu().numpy(),
                batch['rnflt_invalid_fraction'].numpy(),
            ):
                rows.append({
                    'sample_id': str(sid), 'subject_key': str(skey),
                    'y_true': int(label), 'logit': float(logit), 'probability': float(prob),
                    'rnflt_invalid_fraction': float(bad_frac),
                })
    return pd.DataFrame(rows)


def metrics_at_threshold(y_true, probabilities, threshold):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    pred = (p >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'n': int(len(y)),
        'positive': int((y == 1).sum()),
        'negative': int((y == 0).sum()),
        'prevalence': float(np.mean(y)),
        'threshold': float(threshold),
        'auroc': float(roc_auc_score(y, p)),
        'average_precision': float(average_precision_score(y, p)),
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'sensitivity': float(recall_score(y, pred, pos_label=1, zero_division=0)),
        'specificity': float(tn / max(tn + fp, 1)),
        'precision': float(precision_score(y, pred, pos_label=1, zero_division=0)),
        'f1': float(f1_score(y, pred, pos_label=1, zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
        'brier_score': float(brier_score_loss(y, p)),
        'log_loss': float(log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def stratified_bootstrap_ci(y_true, probabilities, threshold, n_bootstrap=N_BOOTSTRAP, seed=20261019):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probabilities, dtype=float)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    if len(pos) == 0 or len(neg) == 0:
        raise RuntimeError('Stratified bootstrap requires both classes in the GDP test set.')
    metric_names = [
        'auroc', 'average_precision', 'accuracy', 'balanced_accuracy', 'sensitivity',
        'specificity', 'precision', 'f1', 'mcc', 'brier_score', 'log_loss',
    ]
    samples = {name: [] for name in metric_names}
    rng = np.random.default_rng(seed)
    for _ in tqdm(range(n_bootstrap), desc='Stratified bootstrap CI'):
        sampled_pos = rng.choice(pos, size=len(pos), replace=True)
        sampled_neg = rng.choice(neg, size=len(neg), replace=True)
        sampled = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled)
        row = metrics_at_threshold(y[sampled], p[sampled], threshold)
        for name in metric_names:
            samples[name].append(row[name])
    return {
        name: {
            'ci95_low': float(np.percentile(values, 2.5)),
            'ci95_high': float(np.percentile(values, 97.5)),
            'valid_bootstrap_replicates': int(len(values)),
        }
        for name, values in samples.items()
    }


test_predictions = predict_dataset(model, test_dataset, 'Harvard-GDP glaucoma test')
test_predictions['prediction'] = (test_predictions['probability'] >= THRESHOLD).astype(int)
y_test = test_predictions['y_true'].to_numpy(dtype=int)
p_test = test_predictions['probability'].to_numpy(dtype=float)

if len(np.unique(y_test)) != 2:
    raise RuntimeError('GDP glaucoma test must contain both glaucoma classes for AUROC/PR evaluation.')
test_metrics = metrics_at_threshold(y_test, p_test, THRESHOLD)
test_ci = stratified_bootstrap_ci(y_test, p_test, THRESHOLD)
for metric, interval in test_ci.items():
    test_metrics[metric + '_ci95_low'] = interval['ci95_low']
    test_metrics[metric + '_ci95_high'] = interval['ci95_high']

test_predictions.to_csv(OUT_DIR / 'gdp_test_predictions.csv', index=False)
(OUT_DIR / 'gdp_test_metrics.json').write_text(json.dumps({
    'status': 'GDP_GLAUCOMA_EXTERNAL_TEST_EVALUATION_COMPLETE',
    'evaluated_model': 'final supervised Harvard-GF combined OCT + RNFLT model',
    'dataset': 'harvard_gdp', 'split': 'glaucoma_split=test',
    'checkpoint_path': str(BEST_PATH), 'checkpoint_sha256': CHECKPOINT_SHA256,
    'dino_checkpoint_sha256': DINO_SHA256,
    'source_threshold': THRESHOLD,
    'source_threshold_method': locked_protocol.get('threshold_selection', 'GF validation lock'),
    'threshold_selected_using_gdp_test': False,
    'gdp_test_used_for_training_or_fine_tuning': False,
    'ssl_exposure_audit': 'PASS: all selected GDP test identifiers are disjoint from the 500 GDP SSL-pool identifiers.',
    'preprocessing_note': 'Same GF model transform: select 12 axis-0 scans, resize each scan to 224x224, center-pad to 224x320, impute invalid RNFLT pixels by nearest valid pixel, and normalize RNFLT with GF-train-only statistics.',
    'bootstrap': {'method': 'stratified bootstrap within each class', 'replicates': N_BOOTSTRAP, 'confidence_level': 0.95, 'seed': 20261019},
    'test_metrics': test_metrics,
    'confidence_intervals': test_ci,
    'clinical_use': 'research-only external evaluation; not a clinically validated diagnostic system',
}, indent=2), encoding='utf-8')
pd.DataFrame([test_metrics]).to_csv(OUT_DIR / 'gdp_test_metrics.csv', index=False)
pd.DataFrame([
    {'metric': k, **v} for k, v in test_ci.items()
]).to_csv(OUT_DIR / 'gdp_test_metrics_ci95.csv', index=False)

print('GDP test metrics at the GF-validation-locked threshold:')
display(pd.DataFrame([test_metrics]).round(4))
print('\n95% stratified-bootstrap confidence intervals:')
display(pd.DataFrame([{'metric': k, **v} for k, v in test_ci.items()]).round(4))
print('Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}'.format(**test_metrics))
print('Predictions saved to:', OUT_DIR / 'gdp_test_predictions.csv')


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'Model AUROC={roc_auc_score(y_test, p_test):.3f}')
ax.plot([0, 1], [0, 1], linestyle='--',color='gray', linewidth=1, label='Chance level (0.500)')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='False Positive Rate', ylabel='True Positive Rate', title='Harvard-GDP Test: ROC')
ax.grid(alpha=0.25); ax.legend(loc='lower right'); fig.tight_layout()
fig.savefig(OUT_DIR / 'roc_test.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'roc_test.pdf', bbox_inches='tight'); plt.show(); plt.close(fig)

# Precision–Recall curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, p_test)
prevalence = float(np.mean(y_test))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(recall_curve, precision_curve,color='darkorange', linewidth=2, label=f'Model Average Precision={average_precision_score(y_test, p_test):.3f}')
ax.axhline(prevalence, linestyle='--', color='gray', linewidth=1, label=f'Positive Prevalence={prevalence:.3f}')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Recall', ylabel='Precision', title='Harvard-GDP Test: Precision–Recall')
ax.grid(alpha=0.25); ax.legend(loc='lower left'); fig.tight_layout()
fig.savefig(OUT_DIR / 'precision_recall_test.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'precision_recall_test.pdf', bbox_inches='tight'); plt.show(); plt.close(fig)


def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return (float('nan'), float('nan'))
    p = successes / total
    denom = 1.0 + z * z / total
    center = (p + z * z / (2.0 * total)) / denom
    half = z * np.sqrt(p * (1.0 - p) / total + z * z / (4.0 * total * total)) / denom
    return float(max(0.0, center - half)), float(min(1.0, center + half))

edges = np.linspace(0.0, 1.0, N_CALIBRATION_BINS + 1)
bin_ids = np.minimum(
    np.searchsorted(edges[1:-1], p_test, side='left'),
    N_CALIBRATION_BINS - 1
)
calibration_rows = []
for bin_id in range(N_CALIBRATION_BINS):
    mask = bin_ids == bin_id
    n_bin = int(mask.sum())
    if n_bin == 0:
        continue
    observed = float(y_test[mask].mean())
    lower, upper = wilson_interval(int(y_test[mask].sum()), n_bin)
    calibration_rows.append({
        'bin': int(bin_id), 'bin_left': float(edges[bin_id]), 'bin_right': float(edges[bin_id + 1]),
        'n': n_bin, 'mean_predicted_probability': float(p_test[mask].mean()),
        'observed_positive_fraction': observed, 'wilson_ci95_low': lower, 'wilson_ci95_high': upper,
    })
calibration_table = pd.DataFrame(calibration_rows)
calibration_table.to_csv(OUT_DIR / 'calibration_bins_test.csv', index=False)

# Calibration curve with 95% Wilson confidence intervals

# Sort bin estimates by mean predicted probability.
calibration_plot = calibration_table.sort_values(
    'mean_predicted_probability'
)

x = calibration_plot['mean_predicted_probability'].to_numpy()
y = calibration_plot['observed_positive_fraction'].to_numpy()
lower_ci = calibration_plot['wilson_ci95_low'].to_numpy()
upper_ci = calibration_plot['wilson_ci95_high'].to_numpy()

fig, ax = plt.subplots(figsize=(7.5, 6.5))

# Perfect calibration reference line.
ax.plot(
    [0, 1], [0, 1],
    linestyle='--',
    color='gray',
    linewidth=1.5,
    label='Perfect calibration'
)

if not calibration_plot.empty:

    # Shaded 95% Wilson confidence interval.
    ax.fill_between(
        x,
        lower_ci,
        upper_ci,
        color='orange',
        alpha=0.20,
        label='95% Wilson confidence interval'
    )

    # Calibration curve in orange.
    ax.plot(
        x, y,
        color='orange',
        marker='o',
        linewidth=2,
        markersize=6,
        label='Model'
    )

    # Error bars for each individual bin.
    ax.errorbar(
        x, y,
        yerr=np.vstack([
            y - lower_ci,
            upper_ci - y
        ]),
        fmt='none',
        ecolor='orange',
        capsize=4,
        elinewidth=1.3
    )

ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel='Mean predicted probability',
    ylabel='Observed positive fraction',
    title='Harvard-GDP Test: Calibration Curve with 95% Confidence Intervals'
)

ax.grid(alpha=0.25)
ax.legend(loc='upper left')
fig.tight_layout()

fig.savefig(
    OUT_DIR / 'calibration_test.png',
    dpi=300,
    bbox_inches='tight'
)
fig.savefig(
    OUT_DIR / 'calibration_test.pdf',
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Confusion matrix at the unchanged GF validation threshold
cm = confusion_matrix(y_test, test_predictions['prediction'].to_numpy(dtype=int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, interpolation='nearest')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Predicted non-glaucoma', 'Predicted glaucoma'],
       yticklabels=['Actual non-glaucoma', 'Actual glaucoma'], title='GDP Test Confusion Matrix')
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(int(value)), ha='center', va='center')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); fig.tight_layout()
fig.savefig(OUT_DIR / 'confusion_matrix_test.png', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)

print('Calibration bins:')
display(calibration_table.round(4))
print('Probability metrics: Brier={:.4f}; log loss={:.4f}'.format(test_metrics['brier_score'], test_metrics['log_loss']))


In [ ]:
def mc_dropout_probabilities(model, dataset, split_name, n_passes=MC_PASSES):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type == 'cuda'))
    batch_draws = []
    ordered_ids, ordered_keys, ordered_labels = [], [], []
    model.eval()
    dropout_modules = [m for m in model.modules() if isinstance(m, nn.Dropout)]
    if not dropout_modules:
        raise RuntimeError('No Dropout modules found; MC Dropout cannot be performed.')
    for module in dropout_modules:
        module.train()
    model.dino_encoder.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f'MC Dropout {split_name} ({n_passes} passes)'):
            scans, rnflt = preprocess_batch(batch, augment=False)
            b, s = scans.shape[:2]
            flat = scans.reshape(b * s, 1, MODEL_SIZE, MODEL_WIDTH)
            feature_chunks = []
            model.dino_encoder.eval()
            with autocast_context():
                for start in range(0, len(flat), SCAN_CHUNK_SIZE):
                    feature_chunks.append(model.dino_encoder(flat[start:start + SCAN_CHUNK_SIZE]))
                oct_features = torch.cat(feature_chunks, dim=0).reshape(b, s, 512)
                rnflt_features = model.rnflt_encoder.features(rnflt)
            positions = torch.linspace(-1.0, 1.0, s, device=oct_features.device, dtype=oct_features.dtype)
            pos_grid = positions.view(1, s, 1).expand(b, -1, -1)
            score_input = torch.cat([oct_features, pos_grid], dim=-1)
            draws = np.empty((n_passes, b), dtype=np.float32)
            for pass_index in range(n_passes):
                with autocast_context():
                    attention_logits = model.scan_attention(score_input).squeeze(-1)
                    weights = torch.softmax(attention_logits.float(), dim=1).to(oct_features.dtype)
                    pooled = torch.sum(oct_features * weights.unsqueeze(-1), dim=1)
                    pooled_position = torch.sum(positions.view(1, -1) * weights, dim=1, keepdim=True)
                    oct_embedding = model.oct_projection(torch.cat([pooled, pooled_position], dim=1))
                    rnflt_embedding = model.rnflt_encoder.projection(rnflt_features)
                    fused = model.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
                    logits = model.glaucoma_head(fused).squeeze(-1).float()
                    prob = torch.sigmoid(logits)
                draws[pass_index] = prob.cpu().numpy()
            batch_draws.append(draws)
            ordered_ids.extend([str(x) for x in batch['sample_id']])
            ordered_keys.extend([str(x) for x in batch['subject_key']])
            ordered_labels.extend(batch['label'].cpu().numpy().astype(int).tolist())
    model.eval()
    probabilities = np.concatenate(batch_draws, axis=1)
    if probabilities.shape != (n_passes, len(dataset)) or not np.isfinite(probabilities).all():
        raise RuntimeError(f'Unexpected or non-finite MC probabilities: {probabilities.shape}.')
    return probabilities, ordered_ids, ordered_keys, np.asarray(ordered_labels, dtype=int)


def binary_entropy(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-7, 1.0 - 1e-7)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))


def make_uncertainty_frame(ids, subject_keys, y, mc_draws, deterministic_predictions):
    det = deterministic_predictions.copy()
    det['sample_id'] = det['sample_id'].astype(str)
    order = pd.DataFrame({'sample_id': [str(x) for x in ids], 'subject_key': [str(x) for x in subject_keys],
                          'y_true_mc': np.asarray(y, dtype=int)})
    if order['sample_id'].duplicated().any() or det['sample_id'].duplicated().any():
        raise RuntimeError('Duplicate sample ids while merging deterministic and MC predictions.')
    if set(order['sample_id']) != set(det['sample_id']):
        raise RuntimeError('Deterministic and MC prediction sample IDs do not match.')
    det = order[['sample_id']].merge(det[['sample_id', 'y_true', 'probability']], on='sample_id', how='left', validate='one_to_one')
    order = order.merge(det, on='sample_id', how='left', validate='one_to_one')
    if not np.array_equal(order['y_true_mc'].to_numpy(dtype=int), order['y_true'].to_numpy(dtype=int)):
        raise RuntimeError('Labels differ between deterministic and MC prediction loaders.')
    p_mc = np.clip(np.asarray(mc_draws, dtype=np.float64), 1e-7, 1.0 - 1e-7)
    mean_p = p_mc.mean(axis=0)
    pred_entropy = binary_entropy(mean_p)
    expected_entropy = binary_entropy(p_mc).mean(axis=0)
    det_p = order['probability'].to_numpy(dtype=float)
    pred = (det_p >= THRESHOLD).astype(int)
    error = (pred != order['y_true'].to_numpy(dtype=int)).astype(int)
    out = order[['sample_id', 'subject_key', 'y_true', 'probability']].copy()
    out = out.rename(columns={'probability': 'det_probability'})
    out['det_prediction'] = pred
    out['error'] = error
    out['mc_mean_probability'] = mean_p
    out['mc_probability_std'] = p_mc.std(axis=0, ddof=0)
    out['mc_predictive_entropy'] = pred_entropy
    out['mc_mutual_information'] = np.maximum(pred_entropy - expected_entropy, 0.0)
    out['mc_predictive_variance'] = p_mc.var(axis=0, ddof=0)
    out['deterministic_entropy'] = binary_entropy(det_p)
    out['threshold_closeness'] = 1.0 - np.abs(det_p - THRESHOLD)
    out['mc_mean_minus_deterministic'] = mean_p - det_p
    return out


# Reuse validation uncertainty cutoffs only if the saved file proves it came from this exact checkpoint.
SCORE_COLUMNS = {
    'MC predictive entropy': 'mc_predictive_entropy',
    'MC mutual information': 'mc_mutual_information',
    'MC predictive variance': 'mc_predictive_variance',
    'Deterministic predictive entropy': 'deterministic_entropy',
    'Distance-to-threshold baseline': 'threshold_closeness',
}
#VAL_UNCERTAINTY_DIR = MODEL_OUT_DIR / 'mc_dropout_uncertainty_analysis'
#VAL_UNCERTAINTY_PATH = VAL_UNCERTAINTY_DIR / 'validation_uncertainty_per_sample.csv'
#VAL_MC_SUMMARY_PATH = VAL_UNCERTAINTY_DIR / 'run_summary.json'
VAL_UNCERTAINTY_DIR = (
    MODEL_OUT_DIR
    / 'sensitivity90_secondary_analysis_02'
    / 'mc_dropout_uncertainty_analysis'
)

VAL_UNCERTAINTY_PATH = (
    VAL_UNCERTAINTY_DIR
    / 'validation_uncertainty_per_sample_sensitivity90.csv'
)

VAL_MC_SUMMARY_PATH = (
    VAL_UNCERTAINTY_DIR
    / 'run_summary_sensitivity90.json'
)
val_uncertainty = None
val_cutoff_source = None

if VAL_UNCERTAINTY_PATH.is_file() and VAL_MC_SUMMARY_PATH.is_file():
    saved_summary = json.loads(VAL_MC_SUMMARY_PATH.read_text(encoding='utf-8'))
    if saved_summary.get('checkpoint_sha256') == CHECKPOINT_SHA256:
        candidate_val = pd.read_csv(VAL_UNCERTAINTY_PATH, dtype={'sample_id': str})
        candidate_val['threshold_closeness'] = 1.0 - np.abs(pd.to_numeric(candidate_val['det_probability'],errors='raise').to_numpy(dtype=float) - THRESHOLD)
        if set(SCORE_COLUMNS.values()).issubset(candidate_val.columns) and len(candidate_val) == 300:
            val_uncertainty = candidate_val.copy()
            val_cutoff_source = 'existing Stage 13 GF-validation MC Dropout outputs, checkpoint hash verified'

if val_uncertainty is None:
    print('No matching saved GF-validation uncertainty file; computing validation reference cutoffs from raw GF validation samples.')
    GF_VAL = manifest.loc[
        manifest['dataset'].eq('harvard_gf') & manifest['protocol_split'].eq('validation')
    ].copy().sort_values('sample_id', kind='stable').reset_index(drop=True)
    if len(GF_VAL) != 300:
        raise RuntimeError(f'Expected 300 GF validation samples to set uncertainty cutoffs; found {len(GF_VAL)}.')
    gf_val_arrays = load_input_arrays(GF_VAL, 'harvard_gf')
    gf_val_dataset = ArrayGlaucomaDataset(gf_val_arrays)
    gf_val_predictions = predict_dataset(model, gf_val_dataset, 'GF validation reference')
    val_draws, val_ids, val_keys, val_y = mc_dropout_probabilities(model, gf_val_dataset, 'GF validation reference')
    val_uncertainty = make_uncertainty_frame(val_ids, val_keys, val_y, val_draws, gf_val_predictions)
    val_cutoff_source = 'computed in this notebook from the 300-sample GF validation split'
    val_uncertainty.to_csv(OUT_DIR / 'gf_validation_uncertainty_reference.csv', index=False)
    del gf_val_arrays, gf_val_dataset, gf_val_predictions, val_draws, val_ids, val_keys, val_y
    gc.collect()

model.eval()
test_draws, test_mc_ids, test_mc_keys, test_mc_y = mc_dropout_probabilities(model, test_dataset, 'GDP test')
test_uncertainty = make_uncertainty_frame(test_mc_ids, test_mc_keys, test_mc_y, test_draws, test_predictions)
test_uncertainty.to_csv(OUT_DIR / 'gdp_test_uncertainty_per_sample.csv', index=False)
np.save(OUT_DIR / 'gdp_test_mc_probability_draws.npy', test_draws.astype(np.float32))

print('MC passes:', MC_PASSES)
print('Validation cutoff source:', val_cutoff_source)
print('Uncertainty columns:', list(test_uncertainty.columns))
display(test_uncertainty.head().round(4))


In [ ]:
# Experiment 1: evaluate whether uncertainty scores rank deterministic classification errors.
y_error = test_uncertainty['error'].to_numpy(dtype=int)
error_prevalence = float(y_error.mean())
error_rows = []
for score_name, score_col in SCORE_COLUMNS.items():
    scores = test_uncertainty[score_col].to_numpy(dtype=float)
    if len(np.unique(y_error)) == 2:
        err_auc = float(roc_auc_score(y_error, scores))
        err_ap = float(average_precision_score(y_error, scores))
    else:
        err_auc, err_ap = float('nan'), float('nan')
    error_rows.append({
        'uncertainty_score': score_name, 'score_column': score_col,
        'error_detection_auroc': err_auc, 'error_detection_average_precision': err_ap,
        'test_error_rate_baseline_for_ap': error_prevalence,
        'n': int(len(test_uncertainty)), 'errors': int(y_error.sum()),
    })
error_detection_table = pd.DataFrame(error_rows)
error_detection_table.to_csv(OUT_DIR / 'error_detection_metrics.csv', index=False)

# Experiment 2: selective risk as increasingly uncertain samples are rejected.
risk_rows = []
aurc_rows = []
for score_name, score_col in SCORE_COLUMNS.items():
    scores = test_uncertainty[score_col].to_numpy(dtype=float)
    order = np.argsort(scores, kind='mergesort')
    ordered_errors = y_error[order]
    n = len(ordered_errors)
    cumulative_risk = np.cumsum(ordered_errors) / np.arange(1, n + 1)
    coverages = np.arange(1, n + 1) / n
    aurc = float(np.trapz(cumulative_risk, coverages))
    aurc_rows.append({'uncertainty_score': score_name, 'discrete_aurc': aurc, 'n': int(n)})
    for target_coverage in COVERAGE_TARGETS:
        n_keep = max(1, int(np.ceil(target_coverage * n)))
        kept_errors = ordered_errors[:n_keep]
        risk_rows.append({
            'uncertainty_score': score_name,
            'target_coverage': float(target_coverage),
            'actual_coverage': float(n_keep / n),
            'n_kept': int(n_keep), 'errors_kept': int(kept_errors.sum()),
            'selective_error_rate': float(kept_errors.mean()),
            'discrete_aurc': aurc,
        })
risk_coverage_table = pd.DataFrame(risk_rows)
aurc_table = pd.DataFrame(aurc_rows)
risk_coverage_table.to_csv(OUT_DIR / 'risk_coverage_test.csv', index=False)
aurc_table.to_csv(OUT_DIR / 'aurc_test.csv', index=False)

# Experiment 3: apply uncertainty cutoffs determined from GF validation to GDP test unchanged.
rng = np.random.default_rng(MC_SEED)
group_rows = []
for score_name, score_col in SCORE_COLUMNS.items():
    val_scores = pd.to_numeric(val_uncertainty[score_col], errors='raise').to_numpy(dtype=float)
    test_scores = test_uncertainty[score_col].to_numpy(dtype=float)
    cutoff = float(np.quantile(val_scores, 1.0 - HIGH_UNCERTAINTY_FRACTION, method='linear'))
    high_mask = test_scores >= cutoff
    test_errors = test_uncertainty['error'].to_numpy(dtype=int)
    high_errors = test_errors[high_mask]
    low_errors = test_errors[~high_mask]
    if len(high_errors) == 0 or len(low_errors) == 0:
        high_rate = float(high_errors.mean()) if len(high_errors) else float('nan')
        low_rate = float(low_errors.mean()) if len(low_errors) else float('nan')
        high_ci = wilson_interval(int(high_errors.sum()), len(high_errors)) if len(high_errors) else (float('nan'), float('nan'))
        low_ci = wilson_interval(int(low_errors.sum()), len(low_errors)) if len(low_errors) else (float('nan'), float('nan'))
        diff, diff_ci = float('nan'), (float('nan'), float('nan'))
    else:
        high_rate, low_rate = float(high_errors.mean()), float(low_errors.mean())
        high_ci = wilson_interval(int(high_errors.sum()), len(high_errors))
        low_ci = wilson_interval(int(low_errors.sum()), len(low_errors))
        diff = high_rate - low_rate
        boot_high = np.mean(rng.binomial(1, high_rate, size=(N_BOOTSTRAP, len(high_errors))), axis=1)
        boot_low = np.mean(rng.binomial(1, low_rate, size=(N_BOOTSTRAP, len(low_errors))), axis=1)
        diff_ci = tuple(np.percentile(boot_high - boot_low, [2.5, 97.5]).astype(float))
    group_rows.append({
        'uncertainty_score': score_name, 'score_column': score_col,
        'validation_cutoff_80th_percentile': cutoff,
        'validation_cutoff_source': val_cutoff_source,
        'test_high_n': int(high_mask.sum()), 'test_low_n': int((~high_mask).sum()),
        'test_high_fraction': float(high_mask.mean()),
        'high_uncertainty_error_rate': high_rate,
        'high_error_ci95_low': high_ci[0], 'high_error_ci95_high': high_ci[1],
        'low_uncertainty_error_rate': low_rate,
        'low_error_ci95_low': low_ci[0], 'low_error_ci95_high': low_ci[1],
        'high_minus_low_error_rate': diff,
        'difference_ci95_low': diff_ci[0], 'difference_ci95_high': diff_ci[1],
    })
high_low_table = pd.DataFrame(group_rows)
high_low_table.to_csv(OUT_DIR / 'high_vs_low_uncertainty_test.csv', index=False)

# Risk-coverage plot
fig, ax = plt.subplots(figsize=(8.5, 6.0))
for score_name, score_col in SCORE_COLUMNS.items():
    scores = test_uncertainty[score_col].to_numpy(dtype=float)
    order = np.argsort(scores, kind='mergesort')
    ordered_errors = y_error[order]
    coverages = np.arange(1, len(ordered_errors) + 1) / len(ordered_errors)
    risks = np.cumsum(ordered_errors) / np.arange(1, len(ordered_errors) + 1)
    ax.plot(coverages, risks, linewidth=1.7, label=score_name)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Coverage (fraction of least-uncertain samples retained)',
       ylabel='Error rate among retained samples', title='Risk–Coverage Harvard-GDP Test')
ax.grid(alpha=0.25); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUT_DIR / 'risk_coverage_test.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'risk_coverage_test.pdf', bbox_inches='tight'); plt.show(); plt.close(fig)

# Error-detection AUROC plot
# Error-detection AUROC plot — same vertical orientation as Stage13
plot_rows = error_detection_table.copy()

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.bar(
    plot_rows['uncertainty_score'],
    plot_rows['error_detection_auroc'],
    color='#C65911'
)

ax.axhline(
    0.5,
    color='gray',
    linestyle='--',
    linewidth=1.2,
    label='Chance (AUROC = 0.5)'
)

ax.set(
    ylim=(0, 1),
    ylabel='Error-detection AUROC',
    title='Error-detection AUROC on External Test'
)

ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()

fig.savefig(
    OUT_DIR / 'error_detection_auroc_test.png',
    dpi=300,
    bbox_inches='tight'
)
fig.savefig(
    OUT_DIR / 'error_detection_auroc_test.pdf',
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print('Experiment 1 — uncertainty score ranking of deterministic errors:')
display(error_detection_table.round(4))
print('\nExperiment 2 — risk–coverage and discrete AURC:')
display(aurc_table.round(4))
display(risk_coverage_table.round(4))
print('\nExperiment 3 — validation-defined high/low uncertainty groups:')
display(high_low_table.round(4))


In [ ]:
# Explainability on a small, explicitly descriptive subset of GDP test cases.
XAI_DIR = OUT_DIR / 'explainability'
XAI_DIR.mkdir(parents=True, exist_ok=True)
XAI_TOP_K = 5
IG_STEPS = 32
TARGET_LAYER = model.dino_encoder.layer4[-1]


def normalize_sample_id(value):
    value = str(value).strip()
    try:
        numeric = float(value)
        if np.isfinite(numeric) and numeric.is_integer():
            return str(int(numeric))
    except (ValueError, TypeError):
        pass
    return value


xai_predictions = test_predictions.copy()
xai_predictions['sample_id'] = xai_predictions['sample_id'].map(normalize_sample_id)
xai_predictions['prediction'] = (xai_predictions['probability'] >= THRESHOLD).astype(int)
xai_positive = xai_predictions.loc[xai_predictions['y_true'].eq(1)].copy()
if xai_positive.empty:
    raise RuntimeError('The GDP test set has no glaucoma-positive cases for descriptive explainability.')
xai_tp = xai_positive.loc[xai_positive['prediction'].eq(1)].sort_values('probability', ascending=False)
if not xai_tp.empty:
    xai_candidates = xai_tp
    xai_selection_basis = 'highest-probability correctly classified glaucoma-positive test cases'
else:
    xai_candidates = xai_positive.sort_values('probability', ascending=False)
    xai_selection_basis = 'highest-probability glaucoma-positive test cases because no true positives were available'
selected_cases = xai_candidates.head(XAI_TOP_K).copy().reset_index(drop=True)
selected_cases['selection_basis'] = xai_selection_basis

_dataset_ids = [normalize_sample_id(x) for x in GDP_TEST_ARRAYS['sample_ids']]
id_to_index = {sample_id: i for i, sample_id in enumerate(_dataset_ids)}
missing_xai_ids = set(selected_cases['sample_id']) - set(id_to_index)
if missing_xai_ids:
    raise RuntimeError(f'Selected XAI sample IDs are absent from the test dataset: {sorted(missing_xai_ids)}')

model.eval()
device = next(model.parameters()).device
left_pad = (MODEL_WIDTH - MODEL_SIZE) // 2


def prepare_xai_sample(dataset_index):
    sample = test_dataset[dataset_index]
    batch = {
        key: value.unsqueeze(0) if torch.is_tensor(value) else [value]
        for key, value in sample.items()
    }
    scans, rnflt = preprocess_batch(batch, augment=False)
    return sample, scans.to(device).float(), rnflt.to(device).float()


def forward_with_oct_gradients(scans, rnflt):
    """Forward the frozen DINO encoder explicitly so layer-4 gradients are available for Grad-CAM."""
    b, s = scans.shape[:2]
    flat = scans.reshape(b * s, 1, MODEL_SIZE, MODEL_WIDTH).detach().requires_grad_(True)
    feature_chunks = []
    for start in range(0, len(flat), SCAN_CHUNK_SIZE):
        x = flat[start:start + SCAN_CHUNK_SIZE]
        x = model.dino_encoder.conv1(x)
        x = model.dino_encoder.bn1(x)
        x = model.dino_encoder.relu(x)
        x = model.dino_encoder.maxpool(x)
        x = model.dino_encoder.layer1(x)
        x = model.dino_encoder.layer2(x)
        x = model.dino_encoder.layer3(x)
        x = model.dino_encoder.layer4(x)
        feature_chunks.append(x)
    feature_maps = torch.cat(feature_chunks, dim=0)
    pooled_features = model.dino_encoder.avgpool(feature_maps).flatten(1)
    features = pooled_features.reshape(b, s, 512)
    positions = torch.linspace(-1.0, 1.0, s, device=features.device, dtype=features.dtype)
    position_grid = positions.view(1, s, 1).expand(b, -1, -1)
    attention_logits = model.scan_attention(torch.cat([features, position_grid], dim=-1)).squeeze(-1)
    attention = torch.softmax(attention_logits.float(), dim=1).to(features.dtype)
    pooled = torch.sum(features * attention.unsqueeze(-1), dim=1)
    pooled_position = torch.sum(positions.view(1, -1) * attention, dim=1, keepdim=True)
    oct_embedding = model.oct_projection(torch.cat([pooled, pooled_position], dim=1))
    rnflt_embedding = model.rnflt_encoder(rnflt)
    fused = model.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
    logit = model.glaucoma_head(fused).squeeze(-1)
    return logit, feature_maps, attention


def make_gradcam(feature_maps, score):
    gradients = torch.autograd.grad(score.sum(), feature_maps, retain_graph=False, create_graph=False)[0]
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * feature_maps).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(MODEL_SIZE, MODEL_WIDTH), mode='bilinear', align_corners=False)[:, 0]
    cam_min = cam.amin(dim=(1, 2), keepdim=True)
    cam_max = cam.amax(dim=(1, 2), keepdim=True)
    return (cam - cam_min) / (cam_max - cam_min + 1e-8)


def rnflt_integrated_gradients(scans, rnflt, steps=IG_STEPS):
    """Compute RNFLT IG while caching the deterministic OCT embedding for efficiency."""
    with torch.no_grad():
        oct_embedding = model.encode_oct(scans)
    baseline = torch.zeros_like(rnflt)
    gradients = torch.zeros_like(rnflt)
    for alpha in torch.linspace(0.0, 1.0, steps, device=rnflt.device):
        x = (baseline + alpha * (rnflt - baseline)).detach().requires_grad_(True)
        rnflt_embedding = model.rnflt_encoder(x)
        fused = model.fusion(torch.cat([oct_embedding, rnflt_embedding], dim=1))
        score = model.glaucoma_head(fused).squeeze(-1)
        gradients += torch.autograd.grad(score.sum(), x, retain_graph=False, create_graph=False)[0].detach()
    attribution = ((rnflt - baseline) * gradients / float(steps)).abs()[0, 0].detach().cpu().numpy()
    attribution /= float(attribution.max()) + 1e-8
    return attribution


def display_scan_canvas(scan_uint8):
    scan = (
        torch.as_tensor(scan_uint8, device=device)
        .float()
        .div(255.0)
        .reshape(1, 1, *scan_uint8.shape)
    )

    h, w = scan.shape[-2:]

    # Preserve the original aspect ratio inside the DINO input canvas.
    scale = min(MODEL_SIZE / h, MODEL_WIDTH / w)
    new_h = max(1, min(MODEL_SIZE, int(round(h * scale))))
    new_w = max(1, min(MODEL_WIDTH, int(round(w * scale))))

    resized = F.interpolate(
        scan,
        size=(new_h, new_w),
        mode='bilinear',
        align_corners=False
    )

    canvas = torch.zeros(
        (1, 1, MODEL_SIZE, MODEL_WIDTH),
        device=device,
        dtype=resized.dtype
    )

    top = (MODEL_SIZE - new_h) // 2
    left = (MODEL_WIDTH - new_w) // 2

    canvas[
        :, :, top:top + new_h, left:left + new_w
    ] = resized

    return canvas[0, 0].detach().cpu().numpy()


xai_records = []
attention_records = []
for _, case in selected_cases.iterrows():
    sample_id = str(case['sample_id'])
    dataset_index = id_to_index[sample_id]
    sample, scans, rnflt = prepare_xai_sample(dataset_index)
    with torch.enable_grad():
        logit, feature_maps, attention = forward_with_oct_gradients(scans, rnflt)
        probability = float(torch.sigmoid(logit).detach().cpu().item())
        gradcam = make_gradcam(feature_maps, logit).detach().cpu().numpy()
    rnflt_ig = rnflt_integrated_gradients(scans, rnflt)

    scan_weights = attention[0].detach().cpu().numpy()
    highest_attention_scan = int(np.argmax(scan_weights))
    raw_scans = sample['oct_uint8'].cpu().numpy()
    scan_image = display_scan_canvas(raw_scans[highest_attention_scan])
    rnflt_display = rnflt[0, 0].detach().cpu().numpy()
    predicted_label = int(probability >= THRESHOLD)
    safe_id = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in sample_id)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
    axes[0].imshow(scan_image, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'OCT B-scan #{highest_attention_scan + 1}')
    axes[0].axis('off')
    axes[1].imshow(scan_image, cmap='gray', vmin=0, vmax=1)
    axes[1].imshow(gradcam[highest_attention_scan], cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[1].set_title('OCT Grad-CAM')
    axes[1].axis('off')
    axes[2].imshow(rnflt_display, cmap='viridis')
    axes[2].set_title('Normalized RNFLT input')
    axes[2].axis('off')
    axes[3].imshow(rnflt_ig, cmap='magma', vmin=0, vmax=1)
    axes[3].set_title('Absolute RNFLT Integrated Gradients')
    axes[3].axis('off')
    fig.suptitle(
        f'Sample {sample_id} | y={int(case["y_true"])} | prediction={predicted_label} | '
        f'P={probability:.3f} | GF threshold={THRESHOLD:.3f}', fontsize=11
    )
    fig.tight_layout()
    fig.savefig(XAI_DIR / f'{safe_id}_gradcam_ig.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8.2, 2.8))
    ax.bar(np.arange(1, N_SCANS + 1), scan_weights)
    ax.set_xlabel('Fixed sampled B-scan position (1–12)')
    ax.set_ylabel('Attention weight')
    ax.set_title(f'Scan attention — {sample_id}')
    ax.set_xticks(np.arange(1, N_SCANS + 1))
    ax.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    fig.savefig(XAI_DIR / f'{safe_id}_scan_attention.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    np.savez_compressed(
        XAI_DIR / f'{safe_id}_gradcam_ig_attention.npz',
        gradcam_by_scan=gradcam.astype(np.float32),
        rnflt_absolute_integrated_gradients=rnflt_ig.astype(np.float32),
        attention_weights=scan_weights.astype(np.float32),
        selected_scan_1based=highest_attention_scan + 1,
        probability=probability,
        locked_threshold=THRESHOLD,
    )
    xai_records.append({
        'sample_id': sample_id,
        'subject_key': str(case['subject_key']),
        'true_label': int(case['y_true']),
        'predicted_label': predicted_label,
        'probability': probability,
        'locked_threshold': THRESHOLD,
        'highest_attention_scan_1based': highest_attention_scan + 1,
        'highest_attention_weight': float(scan_weights[highest_attention_scan]),
        'selection_basis': xai_selection_basis,
    })
    for scan_pos, weight in enumerate(scan_weights, start=1):
        attention_records.append({'sample_id': sample_id, 'scan_position_1based': scan_pos, 'attention_weight': float(weight)})

selected_cases = pd.DataFrame(xai_records)
selected_cases.to_csv(XAI_DIR / 'selected_xai_cases.csv', index=False)
pd.DataFrame(attention_records).to_csv(XAI_DIR / 'scan_attention_weights.csv', index=False)
print(f'Explainability completed for {len(selected_cases)} cases; saved to {XAI_DIR}')
print('Selection basis:', xai_selection_basis)
display(selected_cases)


In [ ]:
# Occlusion sensitivity uses exactly the cases selected above and does not alter the model.
PATCH_SIZE = 32
OCCLUSION_BATCH_SIZE = 4
if MODEL_SIZE % PATCH_SIZE != 0:
    raise ValueError('PATCH_SIZE must divide MODEL_SIZE.')
patch_grid = [
    (y, x, min(y + PATCH_SIZE, MODEL_SIZE), min(x + PATCH_SIZE, MODEL_SIZE))
    for y in range(0, MODEL_SIZE, PATCH_SIZE)
    for x in range(0, MODEL_SIZE, PATCH_SIZE)
]


def occlusion_inputs(dataset_index):
    sample, scans, rnflt = prepare_xai_sample(dataset_index)
    return sample, scans, rnflt


@torch.no_grad()
def batch_logits(scans_batch, rnflt_batch):
    with autocast_context():
        return model(scans_batch, rnflt_batch).float().reshape(-1)


@torch.no_grad()
def occluded_logits(modality, scans, rnflt, scan_index, patch_list, fill_value):
    results = []
    for start in range(0, len(patch_list), OCCLUSION_BATCH_SIZE):
        current = patch_list[start:start + OCCLUSION_BATCH_SIZE]
        n = len(current)
        scans_batch = scans.expand(n, -1, -1, -1, -1).clone()
        rnflt_batch = rnflt.expand(n, -1, -1, -1).clone()
        for j, (y1, x1, y2, x2) in enumerate(current):
            if modality == 'oct':
                scans_batch[j, scan_index, 0, y1:y2, left_pad + x1:left_pad + x2] = fill_value
            elif modality == 'rnflt':
                rnflt_batch[j, 0, y1:y2, x1:x2] = 0.0
            else:
                raise ValueError(f'Unknown modality: {modality}')
        results.extend(batch_logits(scans_batch, rnflt_batch).detach().cpu().numpy().tolist())
    return np.asarray(results, dtype=np.float32)


def patch_delta_map(base_logit, changed_logits):
    deltas = base_logit - changed_logits
    result = np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=np.float32)
    for delta, (y1, x1, y2, x2) in zip(deltas, patch_grid):
        result[y1:y2, x1:x2] = delta
    return result, deltas


occlusion_records = []
model.eval()
for _, case in selected_cases.iterrows():
    sample_id = str(case['sample_id'])
    dataset_index = id_to_index[normalize_sample_id(sample_id)]
    sample, scans, rnflt = occlusion_inputs(dataset_index)
    scan_index = int(case['highest_attention_scan_1based']) - 1
    if not 0 <= scan_index < N_SCANS:
        raise RuntimeError(f'Invalid selected scan index for {sample_id}: {scan_index}')
    base_logit = float(batch_logits(scans, rnflt).item())
    base_probability = float(torch.sigmoid(torch.tensor(base_logit)).item())
    oct_content = scans[0, scan_index, 0, :, left_pad:left_pad + MODEL_SIZE]
    oct_fill = float(oct_content.mean().item())
    oct_changed = occluded_logits('oct', scans, rnflt, scan_index, patch_grid, oct_fill)
    rnflt_changed = occluded_logits('rnflt', scans, rnflt, scan_index, patch_grid, 0.0)
    oct_map, oct_deltas = patch_delta_map(base_logit, oct_changed)
    rnflt_map, rnflt_deltas = patch_delta_map(base_logit, rnflt_changed)
    oct_image = oct_content.detach().cpu().numpy()
    rnflt_image = rnflt[0, 0].detach().cpu().numpy()
    combined_delta = np.concatenate([oct_map.ravel(), rnflt_map.ravel()])
    color_limit = max(float(np.percentile(np.abs(combined_delta), 98)), 1e-6)
    safe_id = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in sample_id)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    axes[0].imshow(oct_image, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'OCT B-scan #{scan_index + 1}')
    axes[0].axis('off')
    axes[1].imshow(oct_image, cmap='gray', vmin=0, vmax=1)
    axes[1].imshow(oct_map, cmap='coolwarm', alpha=0.60, vmin=-color_limit, vmax=color_limit, interpolation='nearest')
    axes[1].set_title('OCT occlusion delta')
    axes[1].axis('off')
    axes[2].imshow(rnflt_image, cmap='viridis')
    axes[2].set_title('Normalized RNFLT input')
    axes[2].axis('off')
    axes[3].imshow(rnflt_image, cmap='viridis')
    overlay = axes[3].imshow(rnflt_map, cmap='coolwarm', alpha=0.60, vmin=-color_limit, vmax=color_limit, interpolation='nearest')
    axes[3].set_title('RNFLT occlusion delta')
    axes[3].axis('off')
    fig.colorbar(overlay, ax=[axes[1], axes[3]], fraction=0.035, pad=0.03, label='Baseline logit − occluded logit')
    fig.suptitle(f'Sample {sample_id} | P(glaucoma)={base_probability:.3f} | patch={PATCH_SIZE}×{PATCH_SIZE}', fontsize=11)
    fig.tight_layout()
    fig.savefig(XAI_DIR / f'{safe_id}_occlusion_sensitivity.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    np.savez_compressed(
        XAI_DIR / f'{safe_id}_occlusion_maps.npz',
        oct_delta_logit=oct_map, rnflt_delta_logit=rnflt_map,
        patch_size=PATCH_SIZE, selected_scan_1based=scan_index + 1,
        baseline_logit=base_logit, baseline_probability=base_probability,
    )
    occlusion_records.append({
        'sample_id': sample_id,
        'selected_scan_1based': scan_index + 1,
        'baseline_logit': base_logit,
        'baseline_probability': base_probability,
        'max_oct_logit_drop': float(np.max(oct_map)),
        'max_rnflt_logit_drop': float(np.max(rnflt_map)),
        'mean_absolute_oct_effect': float(np.mean(np.abs(oct_deltas))),
        'mean_absolute_rnflt_effect': float(np.mean(np.abs(rnflt_deltas))),
        'patch_size': PATCH_SIZE,
        'oct_occlusion_baseline': 'mean intensity within the selected scan content',
        'rnflt_occlusion_baseline': 'zero in GF-train-normalized RNFLT space',
    })

occlusion_summary = pd.DataFrame(occlusion_records)
occlusion_summary.to_csv(XAI_DIR / 'occlusion_sensitivity_summary.csv', index=False)
print('Occlusion sensitivity completed. Results:', XAI_DIR)
display(occlusion_summary)


In [ ]:
summary = {
    'status': 'GDP_GLAUCOMA_EXTERNAL_EVALUATION_MC_DROPOUT_AND_XAI_COMPLETE',
    'dataset': 'harvard_gdp',
    'split': 'glaucoma_split=test',
    'model': 'final supervised Harvard-GF combined OCT + RNFLT v2',
    'checkpoint_path': str(BEST_PATH),
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'dino_sha256': DINO_SHA256,
    'best_gf_epoch': int(checkpoint.get('best_epoch', -1)),
    'locked_threshold_from_gf_validation': THRESHOLD,
    'threshold_method': locked_protocol.get('threshold_selection', 'GF validation lock'),
    'test_count': int(len(test_predictions)),
    'test_positive': int((test_predictions['y_true'] == 1).sum()),
    'test_negative': int((test_predictions['y_true'] == 0).sum()),
    'test_ssl_exposure_audit': 'PASS',
    'gdp_test_used_for_training_finetuning_or_threshold_selection': False,
    'mc_dropout_passes': MC_PASSES,
    'mc_dropout_active_layers': int(sum(isinstance(m, nn.Dropout) for m in model.modules())),
    'validation_cutoff_source': val_cutoff_source,
    'test_error_rate_at_locked_threshold': float(test_uncertainty['error'].mean()),
    'test_metrics': test_metrics,
    'best_error_detection_auroc': float(error_detection_table['error_detection_auroc'].max()) if error_detection_table['error_detection_auroc'].notna().any() else None,
    'lowest_discrete_aurc_score': str(aurc_table.sort_values('discrete_aurc').iloc[0]['uncertainty_score']) if len(aurc_table) else None,
    'preprocessing_caveat': 'GDP B-scans are resized to square 224x224 content to match the GF-trained pipeline; cross-dataset geometry/domain shift may affect performance.',
    'explainability_directory': str(OUT_DIR / 'explainability'),
    'explainability_cases': int(len(selected_cases)),
    'output_directory': str(OUT_DIR),
}
(OUT_DIR / 'final_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(json.dumps({
    'status': summary['status'],
    'test_n': summary['test_count'],
    'test_positive': summary['test_positive'],
    'test_negative': summary['test_negative'],
    'locked_gf_threshold': THRESHOLD,
    'auroc': test_metrics['auroc'],
    'average_precision': test_metrics['average_precision'],
    'sensitivity': test_metrics['sensitivity'],
    'specificity': test_metrics['specificity'],
    'explainability_directory': str(OUT_DIR / 'explainability'),
    'explainability_cases': int(len(selected_cases)),
    'output_directory': str(OUT_DIR),
}, indent=2))
print('\nSaved artifacts:')
for path in sorted(OUT_DIR.iterdir()):
    print(f'{path.name:42s} {path.stat().st_size:>12,d} bytes')


In [ ]:
import os
import shutil
from google.colab import files

source_dir = (
    "/content/drive/MyDrive/backup/glaucoma_oct_research/"
    "results/stage14_gdp_glaucoma_external_eval_gf_v1"
)

zip_path = "/content/stage14_gdp_glaucoma_external_eval_gf_v1.zip"

if not os.path.isdir(source_dir):
    raise FileNotFoundError(
        f"  :\n{source_dir}\n"
        "  Google Drive   Colab mount ‌."
    )

parent_dir = os.path.dirname(source_dir)
folder_name = os.path.basename(source_dir)

shutil.make_archive(
    base_name=zip_path.removesuffix(".zip"),
    format="zip",
    root_dir=parent_dir,
    base_dir=folder_name
)

size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f" ZIP  : {zip_path}")
print(f" : {size_mb:.2f} MB")

files.download(zip_path)

## Final locked operating point and external summary

In [ ]:
from pathlib import Path
import json
import pandas as pd

THRESHOLD = 0.404529
print(f"Final locked threshold: {THRESHOLD:.6f}")

stage13_metrics = OUT_DIR / "best_model_test_metrics.json"
if stage13_metrics.is_file():
    m = json.loads(stage13_metrics.read_text(encoding="utf-8"))
    print("Harvard-GF test metrics at the locked threshold:")
    for k in ["auroc", "average_precision", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "precision", "f1", "mcc"]:
        if k in m:
            print(f"  {k}: {m[k]}")

if 'MODEL_OUT_DIR' in globals():
    stage14_summary = Path(MODEL_OUT_DIR) / "final_summary.json"
    if stage14_summary.is_file():
        s = json.loads(stage14_summary.read_text(encoding="utf-8"))
        print("\nHarvard-GDP external test summary:")
        for k in ["test_n", "test_positive", "test_negative", "locked_gf_threshold", "auroc", "average_precision", "sensitivity", "specificity"]:
            if k in s:
                print(f"  {k}: {s[k]}")